# Bloc 2 — Statistical Learning for Classification & Decision Making
## Apprentissage statistique pour la classification et la prise de décision

Ce bloc étudie les principaux outils de classification statistique et leur utilisation pour la prise de décision.

Pour chaque étape, la démarche suivie sera :

1. mise en pratique,
2. théorie et formules importantes,
3. amélioration guidée par la théorie.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

In [2]:
data_folder = os.path.join(
    os.getcwd(),
    "data_processed"
)

df_model = joblib.load(
    os.path.join(
        data_folder,
        "df_model_block1.joblib"
    )
)

X_block1 = joblib.load(
    os.path.join(
        data_folder,
        "X_block1_final.joblib"
    )
)

y_block1 = joblib.load(
    os.path.join(
        data_folder,
        "y_block1_final.joblib"
    )
)

feature_lists = joblib.load(
    os.path.join(
        data_folder,
        "feature_lists.joblib"
    )
)

split_config = joblib.load(
    os.path.join(
        data_folder,
        "split_config.joblib"
    )
)

In [3]:
features_hd = feature_lists[
    "features_hd"
]

features_hd_advanced = feature_lists[
    "features_hd_advanced"
]

features_stable = feature_lists[
    "features_stable"
]

lasso_selected_features = feature_lists[
    "lasso_selected_features"
]

In [4]:
print("df_model :", df_model.shape)
print("X_block1 :", X_block1.shape)
print("y_block1 :", y_block1.shape)

print("\nFeatures HD :", len(features_hd))
print(
    "Features HD Advanced :",
    len(features_hd_advanced)
)
print(
    "Features Stable :",
    len(features_stable)
)
print(
    "Features Lasso :",
    len(lasso_selected_features)
)

print("\nSplit :", split_config)

df_model : (236254, 378)
X_block1 : (236254, 416)
y_block1 : (236254,)

Features HD : 361
Features HD Advanced : 372
Features Stable : 298
Features Lasso : 244

Split : {'train_end': '2025-01-01', 'validation_start': '2025-01-01', 'validation_end': '2026-01-01', 'test_start': '2026-01-01'}


## Étape 1 — Binary Classification & Decision Thresholds
### Classification binaire et seuil de décision

### Partie A — Mise en pratique

#### A.1 Construction de la cible

L'objectif de cette première étape est de transformer le problème de prévision de consommation en un problème de classification.

On cherchera à prédire si la consommation électrique à un instant donné appartient à une période de forte demande.

On commencera par construire une cible binaire, puis un modèle de classification simple servira de baseline.

La régression logistique sera utilisée comme premier modèle.

### 1.1 Construction d'une cible à trois niveaux de consommation

La consommation est séparée en trois classes :

- classe 0 : faible consommation,
- classe 1 : consommation intermédiaire,
- classe 2 : forte consommation.

Les seuils sont calculés uniquement sur les données d'entraînement afin d'éviter toute fuite d'information.

In [5]:
train_consumption = df_model.loc[
    df_model.index < split_config["train_end"],
    "Consommation (MW)"
]

low_threshold = train_consumption.quantile(
    0.25
)

high_threshold = train_consumption.quantile(
    0.75
)

print(
    "Seuil bas :",
    low_threshold,
    "MW"
)

print(
    "Seuil haut :",
    high_threshold,
    "MW"
)

Seuil bas : 44116.0 MW
Seuil haut : 60824.0 MW


In [6]:
df_model["demand_class"] = 1

C:\Users\thiba\AppData\Local\Temp\ipykernel_28760\898375937.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_model["demand_class"] = 1


In [7]:
df_model.loc[
    df_model["Consommation (MW)"]
    < low_threshold,
    "demand_class"
] = 0

In [8]:
df_model.loc[
    df_model["Consommation (MW)"]
    > high_threshold,
    "demand_class"
] = 2

In [9]:
print(
    df_model["demand_class"].value_counts()
)

print(
    df_model["demand_class"].value_counts(
        normalize=True
    )
)

demand_class
1    118921
0     59550
2     57783
Name: count, dtype: int64
demand_class
1    0.503361
0    0.252059
2    0.244580
Name: proportion, dtype: float64


#### A.2 Variables utilisées

In [10]:
features_classification = [
    "lag_1",
    "lag_48",
    "lag_336",
    "rolling_mean_48",
    "rolling_mean_336",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos"
]

In [11]:
df_classification = df_model[
    features_classification
    + ["demand_class"]
].dropna()

X_class = df_classification[
    features_classification
]

y_class = df_classification[
    "demand_class"
]

In [12]:
print(X_class.shape)
print(y_class.value_counts())

(236254, 11)
demand_class
1    118921
0     59550
2     57783
Name: count, dtype: int64


#### A.3 Split temporel

In [13]:
X_train_class = X_class[
    X_class.index
    < split_config["train_end"]
]

y_train_class = y_class[
    y_class.index
    < split_config["train_end"]
]

X_val_class = X_class[
    (X_class.index >= split_config["validation_start"])
    &
    (X_class.index < split_config["validation_end"])
]

y_val_class = y_class[
    (y_class.index >= split_config["validation_start"])
    &
    (y_class.index < split_config["validation_end"])
]

In [14]:
print("Train :", X_train_class.shape)
print("Validation :", X_val_class.shape)

print("\nRépartition train :")
print(
    y_train_class.value_counts(
        normalize=True
    )
)

print("\nRépartition validation :")
print(
    y_val_class.value_counts(
        normalize=True
    )
)

Train : (210050, 11)
Validation : (17520, 11)

Répartition train :
demand_class
1    0.500005
2    0.249998
0    0.249998
Name: proportion, dtype: float64

Répartition validation :
demand_class
1    0.512671
0    0.294121
2    0.193208
Name: proportion, dtype: float64


#### Conclusion sur le split temporel

La répartition des classes est parfaitement contrôlée sur l'entraînement grâce aux seuils définis à partir des quantiles du train.

Cependant, la distribution change légèrement sur l'année 2025.

La classe de forte consommation devient moins fréquente, tandis que la classe de faible consommation devient plus fréquente.

Cela montre que la distribution temporelle de la consommation évolue et justifie l'utilisation d'une validation chronologique plutôt qu'un découpage aléatoire.

#### A.4 Baseline — Régression logistique multiclasse

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_baseline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000
        )
    )
])

In [16]:
models_folder = os.path.join(
    os.getcwd(),
    "models"
)

os.makedirs(
    models_folder,
    exist_ok=True
)

logistic_path = os.path.join(
    models_folder,
    "logistic_multiclass_baseline.joblib"
)

if os.path.exists(logistic_path):

    logistic_baseline = joblib.load(
        logistic_path
    )

    print("Régression logistique multiclasse chargée")

else:

    logistic_baseline.fit(
        X_train_class,
        y_train_class
    )

    joblib.dump(
        logistic_baseline,
        logistic_path
    )

    print(
        "Régression logistique multiclasse entraînée et sauvegardée"
    )

Régression logistique multiclasse chargée


In [17]:
proba_val = logistic_baseline.predict_proba(
    X_val_class
)

In [18]:
pred_val_class = logistic_baseline.predict(
    X_val_class
)

#### A.5 Première évaluation

In [19]:
accuracy = accuracy_score(
    y_val_class,
    pred_val_class
)

precision_macro = precision_score(
    y_val_class,
    pred_val_class,
    average="macro"
)

recall_macro = recall_score(
    y_val_class,
    pred_val_class,
    average="macro"
)

f1_macro = f1_score(
    y_val_class,
    pred_val_class,
    average="macro"
)

conf_matrix = confusion_matrix(
    y_val_class,
    pred_val_class
)

In [20]:
print("Accuracy :", accuracy)
print("Precision macro :", precision_macro)
print("Recall macro :", recall_macro)
print("F1 macro :", f1_macro)

print(
    "\nMatrice de confusion :\n",
    conf_matrix
)

Accuracy : 0.9561643835616438
Precision macro : 0.9549835133261378
Recall macro : 0.9571692792817293
F1 macro : 0.9560645797084394

Matrice de confusion :
 [[4936  217    0]
 [ 261 8567  154]
 [   0  136 3249]]


In [21]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_val_class,
        pred_val_class,
        target_names=[
            "Faible",
            "Intermédiaire",
            "Forte"
        ]
    )
)

               precision    recall  f1-score   support

       Faible       0.95      0.96      0.95      5153
Intermédiaire       0.96      0.95      0.96      8982
        Forte       0.95      0.96      0.96      3385

     accuracy                           0.96     17520
    macro avg       0.95      0.96      0.96     17520
 weighted avg       0.96      0.96      0.96     17520



#### Conclusion détaillée de la baseline

La régression logistique multiclasse obtient des performances élevées et équilibrées sur les trois classes.

Les scores de précision, rappel et F1 sont tous proches de 0.95–0.96.

Le F1 macro est proche du F1 pondéré, ce qui indique que les performances ne sont pas uniquement dues à la classe intermédiaire, qui est la plus fréquente.

La classe faible, la classe intermédiaire et la classe forte sont toutes correctement reconnues.

Les principales erreurs concernent les observations proches des frontières entre les classes, ce qui est cohérent avec une classification construite à partir de deux seuils de consommation.

Cette baseline constitue donc une référence solide pour les améliorations guidées par la théorie.

### Partie B — Théorie et formules importantes

#### B.1 Régression logistique multiclasse

Dans notre problème, la variable cible possède trois classes :

- classe 0 : faible consommation,
- classe 1 : consommation intermédiaire,
- classe 2 : forte consommation.

La régression logistique multiclasse associe à chaque classe $k$ un score linéaire :

$$
z_k(x) = \beta_k^T x + b_k
$$

avec :

- $x$ : vecteur des variables explicatives,
- $\beta_k$ : coefficients associés à la classe $k$,
- $b_k$ : intercept associé à la classe $k$.

Dans notre cas :

$$
k \in \{0,1,2\}
$$

Le vecteur $x$ contient notamment les lags, les moyennes glissantes et les variables calendaires utilisées par le modèle.


#### B.2 Fonction Softmax

Les scores $z_k(x)$ ne sont pas directement des probabilités.

La fonction Softmax les transforme en probabilités :

$$
P(Y=k \mid X=x)
=
\frac{\exp(z_k(x))}
{\sum_{j=1}^{K}\exp(z_j(x))}
$$

Dans notre cas :

$$
K=3
$$

On obtient donc :

$$
P(Y=0 \mid X)
+
P(Y=1 \mid X)
+
P(Y=2 \mid X)
=
1
$$

Par exemple :

$$
[0.05,\;0.80,\;0.15]
$$

signifie :

- 5 % de probabilité de faible consommation,
- 80 % de probabilité de consommation intermédiaire,
- 15 % de probabilité de forte consommation.

La classe prédite est celle ayant la probabilité la plus élevée :

$$
\hat{Y}
=
\arg\max_k P(Y=k \mid X)
$$


#### B.3 Cross-Entropy / Log-Loss

La régression logistique multiclasse est entraînée en minimisant la cross-entropy.

Pour une observation $i$ :

$$
L_i
=
-\sum_{k=1}^{K}
y_{ik}\log(p_{ik})
$$

où :

- $y_{ik}=1$ si l'observation $i$ appartient à la classe $k$,
- $p_{ik}$ est la probabilité prédite pour cette classe.

Comme une seule classe est correcte, cela revient à :

$$
L_i
=
-\log(p_{i,\text{vraie classe}})
$$

Si la vraie classe est la classe 2 et que le modèle prédit :

$$
[0.05,\;0.10,\;0.85]
$$

la perte est faible, car la vraie classe reçoit une forte probabilité.

À l'inverse, si le modèle prédit :

$$
[0.80,\;0.19,\;0.01]
$$

la perte est beaucoup plus importante.


#### B.4 Matrice de confusion

La matrice de confusion permet d'étudier précisément les erreurs réalisées entre les différentes classes.

On note :

$$
C_{ij}
$$

le nombre d'observations appartenant réellement à la classe $i$ mais prédites comme appartenant à la classe $j$.

Dans notre cas, la matrice obtenue est :

$$
\begin{bmatrix}
4936 & 217 & 0 \\
261 & 8567 & 154 \\
0 & 136 & 3249
\end{bmatrix}
$$

La diagonale représente les bonnes prédictions.

On observe notamment qu'il n'existe aucune confusion directe entre les classes faible et forte :

$$
0 \rightarrow 2 = 0
$$

et :

$$
2 \rightarrow 0 = 0
$$

Les erreurs concernent donc principalement des classes voisines :

$$
0 \leftrightarrow 1
$$

et :

$$
1 \leftrightarrow 2
$$

Ce comportement est cohérent avec la construction ordonnée des trois niveaux de consommation.


#### B.5 Precision, Recall et F1-score

Pour une classe donnée, la précision est définie par :

$$
Precision
=
\frac{TP}{TP+FP}
$$

Elle répond à la question :

> Lorsque le modèle prédit cette classe, à quelle fréquence a-t-il raison ?

Le rappel est défini par :

$$
Recall
=
\frac{TP}{TP+FN}
$$

Il répond à la question :

> Parmi toutes les observations réellement dans cette classe, combien sont détectées ?

Le F1-score combine précision et rappel :

$$
F1
=
2
\frac{Precision \times Recall}
{Precision + Recall}
$$

Il permet de mesurer un compromis entre précision et rappel.

Dans notre baseline, les F1-scores des trois classes sont tous proches de :

$$
0.95 - 0.96
$$

ce qui montre que les performances sont équilibrées entre les trois niveaux de consommation.


#### B.6 Macro-average et Weighted-average

En classification multiclasse, les métriques peuvent être agrégées de différentes manières.

Le macro-average donne le même poids à chaque classe :

$$
Metric_{macro}
=
\frac{1}{K}
\sum_{k=1}^{K}
Metric_k
$$

Chaque classe compte donc autant, indépendamment de sa fréquence.

Le weighted-average pondère chaque classe par son nombre d'observations :

$$
Metric_{weighted}
=
\sum_{k=1}^{K}
\frac{n_k}{n}
Metric_k
$$

Dans notre cas :

$$
F1_{macro} \approx 0.956
$$

et le F1 pondéré est également proche de :

$$
0.96
$$

Le fait que ces deux métriques soient très proches indique que les bonnes performances du modèle ne sont pas uniquement dues à la classe intermédiaire, qui est la plus fréquente.

Les trois classes sont correctement reconnues.

#### B.7 Régularisation de la régression logistique

La régression logistique peut surapprendre lorsque les coefficients deviennent trop grands.

Pour limiter ce phénomène, on ajoute une pénalisation.

Avec une pénalisation L2, l'objectif devient :

$$
\text{CrossEntropy}
+
\lambda
\sum_j \beta_j^2
$$

La régularisation pénalise donc les coefficients de grande amplitude.

Elle permet notamment :

- de limiter le surapprentissage,
- d'améliorer la stabilité du modèle,
- de réduire l'influence de variables très corrélées.

Dans `LogisticRegression` de scikit-learn, la force de régularisation est contrôlée par le paramètre $C$.

On a approximativement :

$$
C \propto \frac{1}{\lambda}
$$

Donc :

- petit $C$ $\rightarrow$ forte régularisation,
- grand $C$ $\rightarrow$ faible régularisation.

Le choix de $C$ pourra être optimisé par validation temporelle.

#### B.8 Importance de la standardisation

Avant d'appliquer une régression logistique régularisée, il est important de standardiser les variables.

La standardisation transforme une variable $x$ selon :

$$
z
=
\frac{x-\mu}{\sigma}
$$

où :

- $\mu$ est la moyenne,
- $\sigma$ est l'écart-type.

Après standardisation, les variables sont exprimées sur des échelles comparables.

Cela est particulièrement important avec la régularisation.

Sans standardisation, une variable de grande amplitude pourrait être pénalisée différemment d'une variable de petite amplitude.

C'est pourquoi notre pipeline utilise :

`StandardScaler -> LogisticRegression`

#### B.9 Structure ordinale des classes

Nos trois classes possèdent un ordre naturel :

$$
\text{Faible}
<
\text{Intermédiaire}
<
\text{Forte}
$$

Cependant, la régression logistique multiclasse classique traite les classes comme des catégories distinctes.

Elle ne prend pas directement en compte le fait que la classe 0 est plus proche de la classe 1 que de la classe 2.

Par exemple, une erreur :

$$
0 \rightarrow 1
$$

et une erreur :

$$
0 \rightarrow 2
$$

sont toutes les deux considérées comme une mauvaise classification.

Pourtant, dans notre problème, une confusion entre faible et forte consommation serait beaucoup plus importante qu'une confusion entre faible et intermédiaire.

Cette structure ordinale pourra être exploitée dans une amélioration du modèle.

#### B.10 Changement de distribution temporelle

La répartition des classes n'est pas exactement la même entre les données d'entraînement et la validation.

Sur l'entraînement :

$$
P(Y=0)\approx 25\%
$$

$$
P(Y=1)\approx 50\%
$$

$$
P(Y=2)\approx 25\%
$$

Sur la validation 2025 :

$$
P(Y=0)\approx 29.4\%
$$

$$
P(Y=1)\approx 51.3\%
$$

$$
P(Y=2)\approx 19.3\%
$$

La classe de forte consommation devient donc moins fréquente en 2025.

Cela signifie que la distribution du problème évolue dans le temps.

Cette observation justifie l'utilisation d'un split temporel plutôt qu'un split aléatoire.

Elle montre également qu'un modèle performant doit être capable de généraliser malgré une évolution de la distribution des classes.

#### Conclusion de la Partie B

La théorie permet de mieux comprendre le fonctionnement de la baseline et d'identifier plusieurs pistes d'amélioration.

La régression logistique multiclasse transforme des scores linéaires en probabilités grâce à la fonction Softmax et apprend ses coefficients en minimisant la cross-entropy.

Les métriques de classification permettent d'évaluer les performances classe par classe.

La théorie suggère maintenant plusieurs améliorations possibles :

- optimiser la force de régularisation $C$,
- enrichir les variables utilisées,
- exploiter la structure ordinale des classes,
- améliorer la règle de décision,
- vérifier la qualité des probabilités prédites.

Ces améliorations seront étudiées dans la Partie C.

### Partie C — Améliorations guidées par la théorie

#### C.1 Optimisation de la régularisation

La baseline utilise la valeur par défaut :

$$
C=1
$$

Dans `LogisticRegression`, un petit $C$ correspond à une régularisation plus forte, tandis qu'un grand $C$ correspond à une régularisation plus faible.

L'objectif est maintenant de déterminer si une autre valeur de $C$ améliore la généralisation du modèle.

Le choix de $C$ doit être effectué uniquement à partir des données d'entraînement afin de conserver la validation 2025 comme évaluation indépendante.

In [22]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score

In [23]:
C_values = [
    0.001,
    0.01,
    0.1,
    0.5,
    1,
    2,
    10,
    100
]

In [24]:
tscv = TimeSeriesSplit(
    n_splits=5
)

cv_results = []

cv_path = os.path.join(
    models_folder,
    "logistic_C_tuning_results.joblib"
)

if os.path.exists(cv_path):

    cv_results = joblib.load(
        cv_path
    )

    print(
        "Résultats du tuning chargés"
    )

else:

    for C in C_values:

        fold_scores = []

        for train_idx, val_idx in tscv.split(
            X_train_class
        ):

            X_cv_train = X_train_class.iloc[
                train_idx
            ]

            X_cv_val = X_train_class.iloc[
                val_idx
            ]

            y_cv_train = y_train_class.iloc[
                train_idx
            ]

            y_cv_val = y_train_class.iloc[
                val_idx
            ]

            model = Pipeline([
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "model",
                    LogisticRegression(
                        C=C,
                        max_iter=2000
                    )
                )
            ])

            model.fit(
                X_cv_train,
                y_cv_train
            )

            pred_cv = model.predict(
                X_cv_val
            )

            score = f1_score(
                y_cv_val,
                pred_cv,
                average="macro"
            )

            fold_scores.append(
                score
            )

        mean_score = np.mean(
        fold_scores
    )

        std_score = np.std(
            fold_scores
        )

        cv_results.append({
            "C": C,
            "mean_f1_macro": mean_score,
            "std_f1_macro": std_score
        })
        print(
            "C =",
            C,
            "| F1 macro =",
            mean_score
        )

    joblib.dump(
        cv_results,
        cv_path
    )

Résultats du tuning chargés


In [25]:
results_C = pd.DataFrame(
    cv_results
)

results_C = results_C.sort_values(
    "mean_f1_macro",
    ascending=False
)

print(results_C)

         C  mean_f1_macro  std_f1_macro
7  100.000       0.953243      0.001726
6   10.000       0.953204      0.001748
5    2.000       0.953131      0.001695
4    1.000       0.953027      0.001691
3    0.500       0.952938      0.001686
2    0.100       0.952429      0.001831
1    0.010       0.944751      0.005497
0    0.001       0.919629      0.008764


In [26]:
best_C = results_C.iloc[
    0
]["C"]

print(
    "Meilleur C :",
    best_C
)

Meilleur C : 100.0


#### Conclusion du tuning de $C$

Les meilleures valeurs de $C$ donnent des performances très proches.

Les scores obtenus sont notamment :

- $C=2$ : F1 macro $\approx 0.95313$,
- $C=10$ : F1 macro $\approx 0.95320$,
- $C=100$ : F1 macro $\approx 0.95324$.

Cependant, les écarts entre ces moyennes sont beaucoup plus faibles que la variabilité observée entre les folds, avec un écart-type d'environ :

$$
0.0017
$$

La différence entre $C=10$ et $C=100$ n'est que d'environ :

$$
0.00004
$$

Il n'est donc pas pertinent de choisir automatiquement la valeur donnant la meilleure moyenne.

On retient :

$$
C=2
$$

car cette valeur offre un excellent compromis entre performance et régularisation.

#### C.2 Réentraînement avec le meilleur compromis de régularisation

In [27]:
best_C = 2

logistic_tuned = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            C=best_C,
            max_iter=2000
        )
    )
])

In [28]:
logistic_tuned_path = os.path.join(
    models_folder,
    "logistic_multiclass_C2.joblib"
)

if os.path.exists(logistic_tuned_path):

    logistic_tuned = joblib.load(
        logistic_tuned_path
    )

    print(
        "Modèle tuned chargé"
    )

else:

    logistic_tuned.fit(
        X_train_class,
        y_train_class
    )

    joblib.dump(
        logistic_tuned,
        logistic_tuned_path
    )

    print(
        "Modèle tuned entraîné et sauvegardé"
    )

Modèle tuned chargé


In [29]:
pred_val_tuned = logistic_tuned.predict(
    X_val_class
)

In [30]:
proba_val_tuned = logistic_tuned.predict_proba(
    X_val_class
)

In [31]:
accuracy_tuned = accuracy_score(
    y_val_class,
    pred_val_tuned
)

precision_macro_tuned = precision_score(
    y_val_class,
    pred_val_tuned,
    average="macro"
)

recall_macro_tuned = recall_score(
    y_val_class,
    pred_val_tuned,
    average="macro"
)

f1_macro_tuned = f1_score(
    y_val_class,
    pred_val_tuned,
    average="macro"
)

In [32]:
print(
    "Accuracy :",
    accuracy_tuned
)

print(
    "Precision macro :",
    precision_macro_tuned
)

print(
    "Recall macro :",
    recall_macro_tuned
)

print(
    "F1 macro :",
    f1_macro_tuned
)

Accuracy : 0.9561073059360731
Precision macro : 0.9550003377727659
Recall macro : 0.9570770160668692
F1 macro : 0.9560279741288147


In [33]:
comparison_C = pd.DataFrame({
    "Model": [
        "Baseline C=1",
        "Tuned C=2"
    ],
    "Accuracy": [
        accuracy,
        accuracy_tuned
    ],
    "Precision_macro": [
        precision_macro,
        precision_macro_tuned
    ],
    "Recall_macro": [
        recall_macro,
        recall_macro_tuned
    ],
    "F1_macro": [
        f1_macro,
        f1_macro_tuned
    ]
})

print(
    comparison_C
)

          Model  Accuracy  Precision_macro  Recall_macro  F1_macro
0  Baseline C=1  0.956164         0.954984      0.957169  0.956065
1     Tuned C=2  0.956107         0.955000      0.957077  0.956028


In [34]:
print(
    classification_report(
        y_val_class,
        pred_val_tuned,
        target_names=[
            "Faible",
            "Intermédiaire",
            "Forte"
        ]
    )
)

               precision    recall  f1-score   support

       Faible       0.95      0.96      0.95      5153
Intermédiaire       0.96      0.95      0.96      8982
        Forte       0.96      0.96      0.96      3385

     accuracy                           0.96     17520
    macro avg       0.96      0.96      0.96     17520
 weighted avg       0.96      0.96      0.96     17520



#### Conclusion sur l'optimisation de la régularisation

Le tuning temporel suggérait que plusieurs valeurs de $C$ donnaient des performances très proches.

Après réentraînement sur l'ensemble des données d'entraînement et évaluation sur la validation 2025, le modèle avec $C=2$ n'améliore pas la baseline.

Les F1 macro sont :

$$
F1_{macro}(C=1) \approx 0.956065
$$

$$
F1_{macro}(C=2) \approx 0.956028
$$

L'écart est négligeable.

La valeur $C=1$ est donc conservée pour la suite, car elle offre des performances au moins aussi bonnes sans complexifier inutilement le modèle.

#### C.3 Enrichissement des variables explicatives

La baseline utilise seulement 11 variables explicatives simples.

Le Bloc 1 a permis de construire un ensemble de variables beaucoup plus riche, comprenant notamment :

- de nombreux lags temporels,
- des statistiques glissantes,
- des variables calendaires,
- des différences temporelles,
- des tendances,
- des écarts aux moyennes locales,
- des interactions calendaires.

L'objectif est maintenant de tester si ces variables améliorent la classification des niveaux de consommation.

Afin de comparer équitablement les modèles, la même régression logistique avec $C=1$ est conservée.

In [35]:
print("X_block1 :", X_block1.shape)
print("y_class :", y_class.shape)

print(
    X_block1.index.min(),
    X_block1.index.max()
)

print(
    y_class.index.min(),
    y_class.index.max()
)

X_block1 : (236254, 416)
y_class : (236254,)
2013-01-07 23:00:00+00:00 2026-06-30 21:30:00+00:00
2013-01-07 23:00:00+00:00 2026-06-30 21:30:00+00:00


In [36]:
y_class_hd = df_model.loc[
    X_block1.index,
    "demand_class"
]

In [37]:
print(
    X_block1.shape,
    y_class_hd.shape
)

print(
    y_class_hd.value_counts(
        normalize=True
    )
)

(236254, 416) (236254,)
demand_class
1    0.503361
0    0.252059
2    0.244580
Name: proportion, dtype: float64


In [38]:
X_train_hd = X_block1[
    X_block1.index
    < split_config["train_end"]
]

y_train_hd = y_class_hd[
    y_class_hd.index
    < split_config["train_end"]
]

X_val_hd = X_block1[
    (X_block1.index >= split_config["validation_start"])
    &
    (X_block1.index < split_config["validation_end"])
]

y_val_hd = y_class_hd[
    (y_class_hd.index >= split_config["validation_start"])
    &
    (y_class_hd.index < split_config["validation_end"])
]

In [39]:
logistic_hd = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            C=1,
            max_iter=3000
        )
    )
])

In [40]:
logistic_hd_path = os.path.join(
    models_folder,
    "logistic_multiclass_high_dim.joblib"
)

if os.path.exists(logistic_hd_path):

    logistic_hd = joblib.load(
        logistic_hd_path
    )

    print(
        "Modèle haute dimension chargé"
    )

else:

    logistic_hd.fit(
        X_train_hd,
        y_train_hd
    )

    joblib.dump(
        logistic_hd,
        logistic_hd_path
    )

    print(
        "Modèle haute dimension entraîné et sauvegardé"
    )

Modèle haute dimension chargé


In [41]:
pred_val_hd = logistic_hd.predict(
    X_val_hd
)

In [42]:
accuracy_hd = accuracy_score(
    y_val_hd,
    pred_val_hd
)

precision_macro_hd = precision_score(
    y_val_hd,
    pred_val_hd,
    average="macro"
)

recall_macro_hd = recall_score(
    y_val_hd,
    pred_val_hd,
    average="macro"
)

f1_macro_hd = f1_score(
    y_val_hd,
    pred_val_hd,
    average="macro"
)


print(
    "Accuracy HD :",
    accuracy_hd
)

print(
    "Precision macro HD :",
    precision_macro_hd
)

print(
    "Recall macro HD :",
    recall_macro_hd
)

print(
    "F1 macro HD :",
    f1_macro_hd
)

Accuracy HD : 0.9889269406392694
Precision macro HD : 0.988668034730563
Recall macro HD : 0.9892001495219875
F1 macro HD : 0.9889302570044536


In [43]:
comparison_features = pd.DataFrame({
    "Model": [
        "Baseline 11 features",
        "High-dimensional features"
    ],
    "Accuracy": [
        accuracy,
        accuracy_hd
    ],
    "Precision_macro": [
        precision_macro,
        precision_macro_hd
    ],
    "Recall_macro": [
        recall_macro,
        recall_macro_hd
    ],
    "F1_macro": [
        f1_macro,
        f1_macro_hd
    ]
})

print(
    comparison_features
)

                       Model  Accuracy  Precision_macro  Recall_macro  \
0       Baseline 11 features  0.956164         0.954984      0.957169   
1  High-dimensional features  0.988927         0.988668      0.989200   

   F1_macro  
0  0.956065  
1  0.988930  


#### Conclusion sur l'enrichissement des variables

L'utilisation des variables haute dimension construites dans le Bloc 1 améliore fortement les performances de la classification.

La baseline avec 11 variables obtenait :

$$
F1_{macro} \approx 0.9561
$$

Le modèle enrichi obtient :

$$
F1_{macro} \approx 0.9889
$$

avec également :

$$
Accuracy \approx 98.89\%
$$

$$
Precision_{macro} \approx 98.87\%
$$

$$
Recall_{macro} \approx 98.92\%
$$

Le gain est donc très supérieur à celui obtenu par le simple tuning de la régularisation.

Cela montre que, dans ce problème, l'amélioration de la représentation des données est beaucoup plus importante que l'ajustement fin de l'hyperparamètre $C$.

Les nombreux lags, statistiques glissantes, variables calendaires et interactions issues du Bloc 1 apportent une information temporelle supplémentaire très utile pour distinguer les trois niveaux de consommation.

#### C.4 Analyse des erreurs du modèle haute dimension

In [44]:
from sklearn.metrics import confusion_matrix, classification_report

conf_matrix_hd = confusion_matrix(
    y_val_hd,
    pred_val_hd
)

print(
    conf_matrix_hd
)

[[5107   46    0]
 [  77 8872   33]
 [   0   38 3347]]


In [45]:
print(
    classification_report(
        y_val_hd,
        pred_val_hd,
        target_names=[
            "Faible",
            "Intermédiaire",
            "Forte"
        ]
    )
)

               precision    recall  f1-score   support

       Faible       0.99      0.99      0.99      5153
Intermédiaire       0.99      0.99      0.99      8982
        Forte       0.99      0.99      0.99      3385

     accuracy                           0.99     17520
    macro avg       0.99      0.99      0.99     17520
 weighted avg       0.99      0.99      0.99     17520



In [46]:
errors_hd = pd.DataFrame({
    "true": y_val_hd.values,
    "pred": pred_val_hd
})

errors_hd["distance"] = (
    errors_hd["true"]
    - errors_hd["pred"]
).abs()

In [47]:
print(
    errors_hd["distance"].value_counts(
        normalize=False
    ).sort_index()
)

print(
    "\nProportions :"
)

print(
    errors_hd["distance"].value_counts(
        normalize=True
    ).sort_index()
)

distance
0    17326
1      194
Name: count, dtype: int64

Proportions :
distance
0    0.988927
1    0.011073
Name: proportion, dtype: float64


In [48]:
severe_errors = (
    errors_hd["distance"] == 2
).sum()

print(
    "Nombre d'erreurs 0 <-> 2 :",
    severe_errors
)

Nombre d'erreurs 0 <-> 2 : 0


In [49]:
ordinal_mae = errors_hd[
    "distance"
].mean()

print(
    "Erreur ordinale moyenne :",
    ordinal_mae
)

Erreur ordinale moyenne : 0.011073059360730594


L'accuracy et le F1 considèrent toutes les erreurs de classification de la même manière.

Cependant, les classes de consommation possèdent un ordre naturel :

$$
0 < 1 < 2
$$

Il est donc pertinent d'étudier également la distance entre la vraie classe et la classe prédite :

$$
d_i = |y_i-\hat{y}_i|
$$

Une erreur entre classes voisines possède une distance de 1, tandis qu'une confusion entre faible et forte consommation possède une distance de 2.

L'erreur ordinale moyenne est définie par :

$$
MAE_{ordinal}
=
\frac{1}{n}
\sum_{i=1}^{n}
|y_i-\hat{y}_i|
$$

Cette métrique complète les métriques classiques en tenant compte de la gravité des erreurs.

#### Conclusion de l'analyse des erreurs

Le modèle haute dimension obtient une très forte performance globale :

$$
Accuracy \approx 98.89\%
$$

La matrice de confusion montre que les erreurs restantes concernent uniquement des classes voisines.

Aucune confusion directe n'est observée entre faible et forte consommation :

$$
0 \leftrightarrow 2 = 0
$$

Sur les $17520$ observations de validation :

- $17326$ sont correctement classées,
- $194$ présentent une erreur de distance 1,
- aucune erreur de distance 2 n'est observée.

L'erreur ordinale moyenne est :

$$
MAE_{ordinal}
\approx
0.0111
$$

Le modèle ne se contente donc pas d'avoir un F1 élevé : ses rares erreurs sont également peu sévères du point de vue ordinal.

#### C.5 Analyse des probabilités et de la calibration

Jusqu'à présent, le modèle a été évalué principalement à partir des classes prédites.

Cependant, une régression logistique produit également des probabilités :

$$
P(Y=0|X),\quad P(Y=1|X),\quad P(Y=2|X)
$$

Un bon modèle probabiliste doit être non seulement précis, mais également bien calibré.

Par exemple, parmi les observations auxquelles le modèle attribue environ 90 % de confiance, environ 90 % devraient effectivement être correctement prédites.

La calibration permet donc de vérifier si la confiance annoncée par le modèle correspond à sa fréquence réelle de succès.

In [50]:
proba_val_hd = logistic_hd.predict_proba(
    X_val_hd
)

print(
    proba_val_hd[:5]
)

[[6.50805735e-42 2.55339202e-04 9.99744661e-01]
 [3.06443351e-39 6.17429201e-03 9.93825708e-01]
 [2.47963527e-39 1.04312904e-02 9.89568710e-01]
 [3.32429019e-38 2.19665112e-02 9.78033489e-01]
 [6.86977018e-35 5.84297666e-01 4.15702334e-01]]


In [51]:
confidence_hd = proba_val_hd.max(
    axis=1
)

correct_hd = (
    pred_val_hd == y_val_hd.values
).astype(int)

In [52]:
print(
    "Confiance moyenne :",
    confidence_hd.mean()
)

print(
    "Accuracy réelle :",
    correct_hd.mean()
)

Confiance moyenne : 0.9835503561253247
Accuracy réelle : 0.9889269406392694


In [53]:
calibration_df = pd.DataFrame({
    "confidence": confidence_hd,
    "correct": correct_hd
})

calibration_df["bin"] = pd.cut(
    calibration_df["confidence"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True
)

In [54]:
calibration_table = (
    calibration_df
    .groupby(
        "bin",
        observed=True
    )
    .agg(
        mean_confidence=(
            "confidence",
            "mean"
        ),
        accuracy=(
            "correct",
            "mean"
        ),
        n=(
            "correct",
            "size"
        )
    )
)

print(
    calibration_table
)

            mean_confidence  accuracy      n
bin                                         
(0.5, 0.6]         0.546868  0.580000    150
(0.6, 0.7]         0.653165  0.666667    195
(0.7, 0.8]         0.750473  0.852814    231
(0.8, 0.9]         0.854513  0.924471    331
(0.9, 1.0]         0.997183  0.999579  16613


##### Expected Calibration Error

On peut résumer l'écart entre confiance et performance réelle par l'Expected Calibration Error, ou ECE.

Pour chaque intervalle de confiance $b$ :

$$
ECE
=
\sum_b
\frac{n_b}{n}
\left|
Accuracy_b
-
Confidence_b
\right|
$$

Une valeur proche de 0 indique une bonne calibration.

In [55]:
ece_hd = np.sum(
    (
        calibration_table["n"]
        / len(calibration_df)
    )
    *
    np.abs(
        calibration_table["accuracy"]
        -
        calibration_table[
            "mean_confidence"
        ]
    )
)

print(
    "ECE :",
    ece_hd
)

ECE : 0.005376584513944725


In [56]:
from sklearn.metrics import log_loss

logloss_hd = log_loss(
    y_val_hd,
    proba_val_hd
)

print(
    "Log-Loss :",
    logloss_hd
)

Log-Loss : 0.02806836538299626


Le Brier Score mesure la distance entre les probabilités prédites et les classes réellement observées.

Dans le cas multiclasse :

$$
Brier
=
\frac{1}{n}
\sum_{i=1}^{n}
\sum_{k=1}^{K}
(p_{ik}-y_{ik})^2
$$

Plus cette valeur est faible, meilleures sont les probabilités prédites.

In [57]:
from sklearn.preprocessing import label_binarize

y_val_onehot = label_binarize(
    y_val_hd,
    classes=[0, 1, 2]
)

brier_hd = np.mean(
    np.sum(
        (
            proba_val_hd
            - y_val_onehot
        ) ** 2,
        axis=1
    )
)

print(
    "Brier Score multiclasse :",
    brier_hd
)

Brier Score multiclasse : 0.01639745401227875


#### Conclusion sur la calibration

Le modèle haute dimension produit des probabilités très bien calibrées.

La confiance moyenne est :

$$
0.9836
$$

tandis que l'accuracy réelle est :

$$
0.9889
$$

Le modèle apparaît donc légèrement sous-confiant en moyenne.

L'Expected Calibration Error est très faible :

$$
ECE \approx 0.0054
$$

ce qui indique que les probabilités annoncées sont très proches des fréquences réelles de succès.

Dans la zone de confiance la plus élevée, qui contient la très grande majorité des observations, le modèle atteint :

$$
Confidence \approx 99.72\%
$$

pour une accuracy réelle d'environ :

$$
99.96\%
$$

La Log-Loss est également faible :

$$
LogLoss \approx 0.0281
$$

ainsi que le Brier Score multiclasse :

$$
Brier \approx 0.0164
$$

Ces résultats indiquent qu'une recalibration supplémentaire n'est pas nécessaire à ce stade.

Le modèle est donc à la fois très précis et fiable du point de vue probabiliste.

#### C.6 Décision ordinale et sensible au coût

La règle de décision classique de la régression logistique choisit simplement la classe ayant la probabilité la plus élevée :

$$
\hat{y}
=
\arg\max_k P(Y=k|X)
$$

Cette règle considère implicitement toutes les erreurs de classification de la même manière.

Or, dans notre problème, les classes possèdent un ordre naturel :

$$
0 < 1 < 2
$$

Il est donc possible d'introduire un coût dépendant de la distance entre la vraie classe et la classe prédite.

On définit :

$$
C(a,k)=|a-k|
$$

où :

- $a$ est la classe choisie,
- $k$ est la vraie classe.

Ainsi :

$$
C(0,1)=1
$$

mais :

$$
C(0,2)=2
$$

Une confusion entre faible et forte consommation est donc considérée comme deux fois plus grave qu'une confusion entre deux classes voisines.

Pour chaque classe candidate $a$, on calcule son coût conditionnel attendu :

$$
R(a|x)
=
\sum_{k=0}^{2}
C(a,k)P(Y=k|X=x)
$$

La décision optimale est :

$$
\hat{y}_{cost}
=
\arg\min_a R(a|x)
$$

Cette règle ne choisit donc plus nécessairement la classe ayant la probabilité maximale.

Elle choisit la classe qui minimise l'erreur ordinale attendue.

In [58]:
import numpy as np

cost_matrix = np.array([
    [0, 1, 2],
    [1, 0, 1],
    [2, 1, 0]
])

print(cost_matrix)

[[0 1 2]
 [1 0 1]
 [2 1 0]]


In [59]:
expected_costs = (
    proba_val_hd
    @ cost_matrix.T
)

In [60]:
print(
    "Probabilités :",
    proba_val_hd[0]
)

print(
    "Coûts attendus :",
    expected_costs[0]
)

Probabilités : [6.50805735e-42 2.55339202e-04 9.99744661e-01]
Coûts attendus : [1.99974466e+00 9.99744661e-01 2.55339202e-04]


In [61]:
pred_val_cost = np.argmin(
    expected_costs,
    axis=1
)

In [62]:
accuracy_cost = accuracy_score(
    y_val_hd,
    pred_val_cost
)

precision_macro_cost = precision_score(
    y_val_hd,
    pred_val_cost,
    average="macro"
)

recall_macro_cost = recall_score(
    y_val_hd,
    pred_val_cost,
    average="macro"
)

f1_macro_cost = f1_score(
    y_val_hd,
    pred_val_cost,
    average="macro"
)

In [63]:
ordinal_mae_cost = np.mean(
    np.abs(
        y_val_hd.values
        - pred_val_cost
    )
)

severe_errors_cost = np.sum(
    np.abs(
        y_val_hd.values
        - pred_val_cost
    ) == 2
)

In [64]:
print(
    "Accuracy cost-sensitive :",
    accuracy_cost
)

print(
    "F1 macro cost-sensitive :",
    f1_macro_cost
)

print(
    "MAE ordinale :",
    ordinal_mae_cost
)

print(
    "Erreurs sévères 0 <-> 2 :",
    severe_errors_cost
)

Accuracy cost-sensitive : 0.9889269406392694
F1 macro cost-sensitive : 0.9889302570044536
MAE ordinale : 0.011073059360730594
Erreurs sévères 0 <-> 2 : 0


In [65]:
comparison_decision = pd.DataFrame({
    "Decision rule": [
        "Argmax classique",
        "Coût ordinal"
    ],
    "Accuracy": [
        accuracy_hd,
        accuracy_cost
    ],
    "F1_macro": [
        f1_macro_hd,
        f1_macro_cost
    ],
    "Ordinal_MAE": [
        ordinal_mae,
        ordinal_mae_cost
    ],
    "Severe_errors": [
        severe_errors,
        severe_errors_cost
    ]
})

print(
    comparison_decision
)

      Decision rule  Accuracy  F1_macro  Ordinal_MAE  Severe_errors
0  Argmax classique  0.988927   0.98893     0.011073              0
1      Coût ordinal  0.988927   0.98893     0.011073              0


#### Conclusion sur la décision sensible au coût

La règle de décision ordinale donne exactement les mêmes performances que la règle classique basée sur l'argmax.

Les deux méthodes obtiennent :

$$
Accuracy \approx 0.98893
$$

$$
F1_{macro} \approx 0.98893
$$

$$
MAE_{ordinal} \approx 0.01107
$$

avec :

$$
0
$$

erreur sévère entre les classes faible et forte.

Le modèle produit déjà des probabilités très concentrées autour de la classe correcte. Dans ce contexte, la classe ayant la probabilité maximale correspond également à la décision minimisant le coût ordinal attendu.

La prise en compte explicite du coût ordinal n'améliore donc pas les performances, mais elle permet de formaliser le problème dans le cadre de la théorie statistique de la décision.

#### C.7 Comparaison finale des améliorations

In [66]:
final_comparison_step1 = pd.DataFrame({
    "Model": [
        "Baseline - 11 features",
        "Tuned Logistic - C=2",
        "High-dimensional Logistic",
        "High-dimensional + Ordinal Decision"
    ],
    "Accuracy": [
        accuracy,
        accuracy_tuned,
        accuracy_hd,
        accuracy_cost
    ],
    "F1_macro": [
        f1_macro,
        f1_macro_tuned,
        f1_macro_hd,
        f1_macro_cost
    ]
})

print(final_comparison_step1)

                                 Model  Accuracy  F1_macro
0               Baseline - 11 features  0.956164  0.956065
1                 Tuned Logistic - C=2  0.956107  0.956028
2            High-dimensional Logistic  0.988927  0.988930
3  High-dimensional + Ordinal Decision  0.988927  0.988930


In [67]:
print(
    "ECE :",
    ece_hd
)

print(
    "Log-Loss :",
    logloss_hd
)

print(
    "Brier Score :",
    brier_hd
)

print(
    "Ordinal MAE :",
    ordinal_mae
)

print(
    "Erreurs sévères :",
    severe_errors
)

ECE : 0.005376584513944725
Log-Loss : 0.02806836538299626
Brier Score : 0.01639745401227875
Ordinal MAE : 0.011073059360730594
Erreurs sévères : 0


#### Conclusion de la Partie C

Plusieurs améliorations guidées par la théorie ont été étudiées.

Le tuning du paramètre de régularisation $C$ n'a apporté aucun gain significatif par rapport à la valeur initiale $C=1$.

En revanche, l'enrichissement des variables explicatives à partir du travail réalisé dans le Bloc 1 a fortement amélioré les performances.

Le F1 macro passe de :

$$
0.9561
$$

pour la baseline à :

$$
0.9889
$$

pour le modèle haute dimension.

L'analyse des erreurs montre également que toutes les erreurs restantes concernent uniquement des classes voisines et qu'aucune confusion directe entre faible et forte consommation n'est observée.

Le modèle présente également une très bonne qualité probabiliste avec :

$$
ECE \approx 0.0054
$$

$$
LogLoss \approx 0.0281
$$

$$
Brier \approx 0.0164
$$

Enfin, l'introduction d'une règle de décision tenant compte du caractère ordinal des classes ne modifie pas les prédictions, car les probabilités produites par le modèle sont déjà suffisamment discriminantes.

L'amélioration principale de cette étape provient donc de l'ingénierie et de l'enrichissement des variables plutôt que du tuning fin des hyperparamètres ou de la modification de la règle de décision.

### Conclusion générale de l'Étape 1

Cette première étape a étudié la classification des niveaux de consommation à l'aide d'une régression logistique multiclasse.

Une première baseline utilisant 11 variables obtenait :

$$
F1_{macro} \approx 0.9561
$$

L'optimisation de la régularisation n'a pas apporté d'amélioration significative.

En revanche, l'utilisation des variables haute dimension construites dans le Bloc 1 a permis d'atteindre :

$$
F1_{macro} \approx 0.9889
$$

et :

$$
Accuracy \approx 98.89\%
$$

Les rares erreurs restantes concernent uniquement des classes voisines et aucune confusion directe entre faible et forte consommation n'a été observée.

L'analyse probabiliste a également montré que le modèle est très bien calibré.

Cette étape montre que l'enrichissement de la représentation des données apporte ici un gain beaucoup plus important que le tuning fin des hyperparamètres.

Le modèle logistique haute dimension constitue désormais la référence à battre dans les prochaines étapes.

## Étape 2 — Tree-Based Classification & Random Forests
### Classification par arbres et méthodes de bagging

### Partie A — Mise en pratique

#### A.1 Baseline — Decision Tree Classifier

Un arbre de décision est utilisé comme première méthode non linéaire.

Contrairement à la régression logistique, il peut apprendre automatiquement des relations non linéaires et des interactions entre les variables.

Les mêmes variables haute dimension que précédemment sont utilisées afin de permettre une comparaison équitable.

In [68]:
from sklearn.tree import DecisionTreeClassifier

tree_baseline = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=20,
    random_state=42
)

In [69]:
tree_path = os.path.join(
    models_folder,
    "decision_tree_baseline_block2.joblib"
)

if os.path.exists(tree_path):

    tree_baseline = joblib.load(
        tree_path
    )

    print("Decision Tree chargé")

else:

    tree_baseline.fit(
        X_train_hd,
        y_train_hd
    )

    joblib.dump(
        tree_baseline,
        tree_path
    )

    print(
        "Decision Tree entraîné et sauvegardé"
    )

Decision Tree chargé


In [70]:
pred_val_tree = tree_baseline.predict(
    X_val_hd
)

In [71]:
accuracy_tree = accuracy_score(
    y_val_hd,
    pred_val_tree
)

precision_macro_tree = precision_score(
    y_val_hd,
    pred_val_tree,
    average="macro"
)

recall_macro_tree = recall_score(
    y_val_hd,
    pred_val_tree,
    average="macro"
)

f1_macro_tree = f1_score(
    y_val_hd,
    pred_val_tree,
    average="macro"
)

In [72]:
print(
    "Accuracy Tree :",
    accuracy_tree
)

print(
    "Precision macro Tree :",
    precision_macro_tree
)

print(
    "Recall macro Tree :",
    recall_macro_tree
)

print(
    "F1 macro Tree :",
    f1_macro_tree
)

Accuracy Tree : 0.9746004566210046
Precision macro Tree : 0.9744250969212921
Recall macro Tree : 0.9744642006886227
F1 macro Tree : 0.9744445919495898


In [73]:
conf_matrix_tree = confusion_matrix(
    y_val_hd,
    pred_val_tree
)

print(
    conf_matrix_tree
)

[[5015  138    0]
 [ 135 8760   87]
 [   0   85 3300]]


In [74]:
print(
    classification_report(
        y_val_hd,
        pred_val_tree,
        target_names=[
            "Faible",
            "Intermédiaire",
            "Forte"
        ]
    )
)

               precision    recall  f1-score   support

       Faible       0.97      0.97      0.97      5153
Intermédiaire       0.98      0.98      0.98      8982
        Forte       0.97      0.97      0.97      3385

     accuracy                           0.97     17520
    macro avg       0.97      0.97      0.97     17520
 weighted avg       0.97      0.97      0.97     17520



In [75]:
comparison_tree = pd.DataFrame({
    "Model": [
        "Logistic Regression HD",
        "Decision Tree"
    ],
    "Accuracy": [
        accuracy_hd,
        accuracy_tree
    ],
    "F1_macro": [
        f1_macro_hd,
        f1_macro_tree
    ]
})

print(
    comparison_tree
)

                    Model  Accuracy  F1_macro
0  Logistic Regression HD  0.988927  0.988930
1           Decision Tree  0.974600  0.974445


#### Conclusion de la baseline Decision Tree

Le Decision Tree obtient de bonnes performances :

$$
Accuracy \approx 97.46\%
$$

$$
F1_{macro} \approx 0.9744
$$

Cependant, il reste moins performant que la régression logistique haute dimension :

$$
F1_{macro,\ Logistic} \approx 0.9889
$$

L'écart est d'environ :

$$
0.0145
$$

Le Decision Tree ne commet aucune confusion directe entre les classes faible et forte, mais il réalise davantage d'erreurs entre classes voisines.

Ce résultat montre qu'un modèle non linéaire n'est pas automatiquement supérieur à un modèle linéaire lorsque les variables explicatives sont déjà très riches.

Le Decision Tree sera néanmoins utile comme modèle de base pour introduire les méthodes d'ensemble, notamment le Random Forest.

#### A.2 Random Forest Classifier

Un Random Forest combine plusieurs arbres de décision entraînés sur des sous-échantillons des données et des sous-ensembles aléatoires de variables.

L'objectif principal est de réduire la variance d'un arbre unique tout en conservant sa capacité à modéliser des relations non linéaires.

Chaque arbre fournit une prédiction, puis la classe finale est choisie par vote majoritaire.

In [76]:
from sklearn.ensemble import RandomForestClassifier

rf_baseline = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

In [77]:
rf_path = os.path.join(
    models_folder,
    "random_forest_baseline_block2.joblib"
)

if os.path.exists(rf_path):

    rf_baseline = joblib.load(
        rf_path
    )

    print("Random Forest chargé")

else:

    rf_baseline.fit(
        X_train_hd,
        y_train_hd
    )

    joblib.dump(
        rf_baseline,
        rf_path
    )

    print(
        "Random Forest entraîné et sauvegardé"
    )

Random Forest chargé


In [78]:
pred_val_rf = rf_baseline.predict(
    X_val_hd
)

In [79]:
accuracy_rf = accuracy_score(
    y_val_hd,
    pred_val_rf
)

precision_macro_rf = precision_score(
    y_val_hd,
    pred_val_rf,
    average="macro"
)

recall_macro_rf = recall_score(
    y_val_hd,
    pred_val_rf,
    average="macro"
)

f1_macro_rf = f1_score(
    y_val_hd,
    pred_val_rf,
    average="macro"
)

In [80]:
print(
    "Accuracy RF :",
    accuracy_rf
)

print(
    "Precision macro RF :",
    precision_macro_rf
)

print(
    "Recall macro RF :",
    recall_macro_rf
)

print(
    "F1 macro RF :",
    f1_macro_rf
)

Accuracy RF : 0.9759132420091324
Precision macro RF : 0.9757726299061101
Recall macro RF : 0.9754548975110505
F1 macro RF : 0.9756101203064601


In [81]:
conf_matrix_rf = confusion_matrix(
    y_val_hd,
    pred_val_rf
)

print(
    conf_matrix_rf
)

[[5040  113    0]
 [ 137 8767   78]
 [   0   94 3291]]


In [82]:
print(
    classification_report(
        y_val_hd,
        pred_val_rf,
        target_names=[
            "Faible",
            "Intermédiaire",
            "Forte"
        ]
    )
)

               precision    recall  f1-score   support

       Faible       0.97      0.98      0.98      5153
Intermédiaire       0.98      0.98      0.98      8982
        Forte       0.98      0.97      0.97      3385

     accuracy                           0.98     17520
    macro avg       0.98      0.98      0.98     17520
 weighted avg       0.98      0.98      0.98     17520



In [83]:
comparison_rf = pd.DataFrame({
    "Model": [
        "Logistic Regression HD",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_hd,
        accuracy_tree,
        accuracy_rf
    ],
    "F1_macro": [
        f1_macro_hd,
        f1_macro_tree,
        f1_macro_rf
    ]
})

print(
    comparison_rf
)

                    Model  Accuracy  F1_macro
0  Logistic Regression HD  0.988927  0.988930
1           Decision Tree  0.974600  0.974445
2           Random Forest  0.975913  0.975610


#### Conclusion de la baseline Random Forest

Le Random Forest améliore légèrement les performances du Decision Tree.

Les résultats obtenus sont :

$$
F1_{macro,\ Tree} \approx 0.9744
$$

$$
F1_{macro,\ RandomForest} \approx 0.9756
$$

Le gain existe mais reste modéré.

La régression logistique haute dimension conserve néanmoins une avance nette :

$$
F1_{macro,\ Logistic} \approx 0.9889
$$

Le Random Forest réduit donc une partie de la variance d'un arbre unique, mais ne parvient pas à surpasser le modèle linéaire enrichi.

Ce résultat suggère que les variables construites dans le Bloc 1 rendent déjà le problème fortement séparable, ce qui favorise la régression logistique.

### Partie B — Théorie et formules importantes

#### B.1 Principe d'un arbre de décision

Un arbre de décision découpe progressivement l'espace des variables en régions de plus en plus homogènes.

À chaque nœud, l'algorithme cherche une règle du type :

$$
x_j < s
$$

où :

- $x_j$ est une variable,
- $s$ est un seuil.

Cette règle sépare les observations en deux groupes.

L'objectif est de choisir la variable et le seuil qui produisent les groupes les plus purs possible.

Dans un problème de classification, un nœud est considéré comme pur si la majorité des observations qu'il contient appartiennent à la même classe.


#### B.2 Impureté de Gini

L'un des critères les plus utilisés pour mesurer l'impureté d'un nœud est l'indice de Gini.

Pour un nœud contenant $K$ classes :

$$
Gini
=
1
-
\sum_{k=1}^{K}
p_k^2
$$

où $p_k$ représente la proportion d'observations de la classe $k$ dans le nœud.

Dans notre cas :

$$
K=3
$$

Si un nœud contient uniquement des observations de la classe 1 :

$$
p_1=1
$$

et les autres probabilités sont nulles.

Alors :

$$
Gini=0
$$

Le nœud est parfaitement pur.

À l'inverse, si les trois classes sont réparties de manière égale :

$$
p_0=p_1=p_2=\frac{1}{3}
$$

alors :

$$
Gini
=
1
-
3\left(\frac{1}{3}\right)^2
=
\frac{2}{3}
$$

L'impureté est alors beaucoup plus élevée.


#### B.3 Choix d'un split

Pour chaque split candidat, l'arbre compare l'impureté avant et après la séparation.

On cherche à maximiser la diminution d'impureté :

$$
\Delta I
=
I_{parent}
-
\left(
\frac{n_L}{n}I_L
+
\frac{n_R}{n}I_R
\right)
$$

où :

- $I_{parent}$ est l'impureté du nœud parent,
- $I_L$ est l'impureté du nœud gauche,
- $I_R$ est l'impureté du nœud droit,
- $n_L$ et $n_R$ sont les nombres d'observations dans les deux sous-nœuds.

L'arbre sélectionne le split qui produit la plus forte réduction d'impureté.


#### B.4 Entropie

Une autre mesure possible est l'entropie :

$$
H
=
-
\sum_{k=1}^{K}
p_k\log(p_k)
$$

Comme le Gini, l'entropie vaut 0 lorsque le nœud est parfaitement pur.

Plus les classes sont mélangées, plus l'entropie augmente.

En pratique, Gini et entropie donnent souvent des arbres assez proches.

Le critère Gini est généralement légèrement plus simple à calculer.


#### B.5 Profondeur de l'arbre et surapprentissage

La profondeur correspond au nombre maximal de décisions successives entre la racine et une feuille.

Un arbre peu profond produit des règles simples.

Il peut avoir :

$$
Biais \uparrow
$$

et :

$$
Variance \downarrow
$$

À l'inverse, un arbre très profond peut apprendre des règles extrêmement spécifiques au jeu d'entraînement.

Il peut alors avoir :

$$
Biais \downarrow
$$

mais :

$$
Variance \uparrow
$$

Un arbre complètement développé peut donc surapprendre fortement les données.

Dans notre baseline, nous avons imposé :

`max_depth = 10`

afin de limiter cette variance.


#### B.6 min_samples_leaf

Le paramètre `min_samples_leaf` fixe le nombre minimal d'observations dans une feuille.

Une petite valeur permet à l'arbre de créer des régions très spécifiques.

Cela augmente sa flexibilité mais également son risque de surapprentissage.

Une valeur plus grande impose des régions contenant davantage d'observations et produit donc un arbre plus régularisé.

Dans notre baseline :

`min_samples_leaf = 20`


#### B.7 Pourquoi un arbre possède souvent une variance élevée

Les arbres sont sensibles aux données utilisées pour leur apprentissage.

Une petite modification du jeu d'entraînement peut modifier le premier split.

Cette modification peut ensuite changer les splits suivants et produire un arbre très différent.

On dit donc qu'un arbre possède généralement une variance élevée.

Cette propriété explique l'intérêt du bagging et des Random Forests.


#### B.8 Bagging

Le bagging signifie :

`Bootstrap Aggregating`

L'idée est d'entraîner plusieurs modèles sur différentes versions du jeu de données.

Pour chaque arbre :

1. on tire aléatoirement des observations avec remise dans les données d'entraînement ;
2. on entraîne un arbre sur cet échantillon bootstrap ;
3. on répète l'opération pour plusieurs arbres.

En classification, les prédictions sont ensuite agrégées.

Une version simplifiée du vote est :

$$
\hat y
=
\operatorname{mode}
\left(
\hat y_1,
\hat y_2,
\dots,
\hat y_B
\right)
$$

où $B$ représente le nombre d'arbres.

L'objectif principal du bagging est de réduire la variance.


#### B.9 Pourquoi moyenner plusieurs modèles réduit la variance

Supposons que plusieurs estimateurs aient la même variance :

$$
Var(\hat f_b)=\sigma^2
$$

Si leurs erreurs étaient complètement indépendantes, la moyenne de $B$ modèles aurait approximativement :

$$
Var\left(
\frac{1}{B}
\sum_{b=1}^{B}
\hat f_b
\right)
=
\frac{\sigma^2}{B}
$$

La variance diminuerait donc lorsque le nombre de modèles augmente.

En pratique, les arbres ne sont pas indépendants.

Ils sont corrélés car ils utilisent les mêmes variables et des données proches.

Le Random Forest ajoute donc un deuxième mécanisme pour réduire cette corrélation.


#### B.10 Random Forest et sélection aléatoire des variables

Dans un Random Forest, chaque arbre utilise non seulement un échantillon bootstrap différent, mais chaque split ne considère également qu'un sous-ensemble aléatoire des variables.

Avec :

`max_features = "sqrt"`

en classification, environ :

$$
\sqrt{p}
$$

variables sont proposées à chaque split, où $p$ est le nombre total de variables.

Cette stratégie empêche les variables les plus fortes d'être systématiquement sélectionnées dans tous les arbres.

Les arbres deviennent donc moins corrélés.

La combinaison :

$$
Bootstrap
+
Random\ Feature\ Selection
$$

permet généralement de réduire la variance davantage qu'un simple arbre.


#### B.11 Nombre d'arbres

Le paramètre :

`n_estimators`

correspond au nombre d'arbres utilisés.

Dans notre baseline :

$$
B=200
$$

Augmenter le nombre d'arbres permet généralement de stabiliser les prédictions.

Contrairement à la profondeur d'un arbre, augmenter fortement le nombre d'arbres ne provoque généralement pas directement de surapprentissage.

Le principal coût supplémentaire est computationnel :

- temps d'entraînement plus long,
- prédiction plus lente,
- consommation mémoire plus importante.


#### B.12 Biais-variance : Decision Tree vs Random Forest

Un Decision Tree possède généralement :

$$
Biais \ faible
$$

mais :

$$
Variance \ élevée
$$

Le Random Forest moyenne plusieurs arbres.

Il cherche principalement à réduire :

$$
Variance
$$

sans augmenter excessivement :

$$
Biais
$$

C'est précisément ce que l'on observe dans nos résultats :

$$
F1_{Tree} \approx 0.9744
$$

$$
F1_{RandomForest} \approx 0.9756
$$

Le Random Forest améliore légèrement les performances de l'arbre unique.


#### B.13 Pourquoi le Random Forest reste inférieur à la régression logistique

Notre régression logistique haute dimension obtient :

$$
F1_{macro} \approx 0.9889
$$

contre :

$$
F1_{macro} \approx 0.9756
$$

pour le Random Forest.

Cela ne signifie pas que la régression logistique est toujours supérieure aux Random Forests.

Dans notre problème, les nombreuses variables construites dans le Bloc 1 semblent déjà rendre les classes presque linéairement séparables.

Une grande partie de la non-linéarité temporelle a déjà été incorporée directement dans les features :

- lags,
- statistiques glissantes,
- différences,
- tendances,
- variables calendaires,
- interactions.

La régression logistique peut donc exploiter efficacement cette représentation riche.

Le Random Forest doit quant à lui reconstruire ces frontières à partir de nombreux splits successifs.


#### Conclusion de la Partie B

Les arbres de décision construisent des règles successives afin de créer des régions homogènes dans l'espace des variables.

Ils sont capables de modéliser naturellement des relations non linéaires et des interactions, mais possèdent souvent une variance élevée.

Le Random Forest réduit cette variance en combinant deux mécanismes :

$$
Bootstrap
+
Randomisation\ des\ variables
$$

L'agrégation de nombreux arbres produit un modèle plus stable qu'un arbre individuel.

Dans notre expérience, le Random Forest améliore légèrement le Decision Tree mais reste inférieur à la régression logistique haute dimension.

La théorie suggère maintenant plusieurs possibilités d'amélioration :

- ajuster la profondeur des arbres,
- ajuster `min_samples_leaf`,
- augmenter ou modifier le nombre d'arbres,
- ajuster `max_features`,
- étudier l'importance des variables,
- comparer les performances et la stabilité des différents modèles.

Ces points seront étudiés dans la Partie C.

### Partie C — Améliorations guidées par la théorie

#### C.1 Contrôle du compromis biais-variance

La théorie montre que les performances d'un arbre dépendent fortement de sa complexité.

Deux paramètres sont particulièrement importants :

- `max_depth` : profondeur maximale de l'arbre ;
- `min_samples_leaf` : nombre minimal d'observations dans une feuille.

Une profondeur trop faible peut conduire à du sous-apprentissage, tandis qu'une profondeur trop élevée peut augmenter la variance.

De même, une petite valeur de `min_samples_leaf` permet des règles très spécifiques, alors qu'une valeur plus élevée régularise davantage les arbres.

On commence donc par rechercher un compromis entre ces deux paramètres à l'aide d'une validation temporelle.

In [84]:
from sklearn.model_selection import TimeSeriesSplit

tscv_rf = TimeSeriesSplit(
    n_splits=3
)

In [85]:
depth_values = [
    8,
    12,
    16,
    20,
    None
]

leaf_values = [
    5,
    10,
    20,
    50
]

In [86]:
rf_tuning_path = os.path.join(
    models_folder,
    "rf_depth_leaf_tuning.joblib"
)

In [87]:
if os.path.exists(rf_tuning_path):

    rf_tuning_results = joblib.load(
        rf_tuning_path
    )

    print(
        "Résultats Random Forest chargés"
    )

else:

    rf_tuning_results = []

    for depth in depth_values:

        for leaf in leaf_values:

            fold_scores = []

            for train_idx, val_idx in tscv_rf.split(
                X_train_hd
            ):

                X_cv_train = X_train_hd.iloc[
                    train_idx
                ]

                X_cv_val = X_train_hd.iloc[
                    val_idx
                ]

                y_cv_train = y_train_hd.iloc[
                    train_idx
                ]

                y_cv_val = y_train_hd.iloc[
                    val_idx
                ]

                model_rf = RandomForestClassifier(
                    n_estimators=100,
                    max_depth=depth,
                    min_samples_leaf=leaf,
                    max_features="sqrt",
                    n_jobs=-1,
                    random_state=42
                )

                model_rf.fit(
                    X_cv_train,
                    y_cv_train
                )

                pred_cv = model_rf.predict(
                    X_cv_val
                )

                score = f1_score(
                    y_cv_val,
                    pred_cv,
                    average="macro"
                )

                fold_scores.append(
                    score
                )

            mean_score = np.mean(
                fold_scores
            )

            std_score = np.std(
                fold_scores
            )

            rf_tuning_results.append({
                "max_depth": depth,
                "min_samples_leaf": leaf,
                "mean_f1_macro": mean_score,
                "std_f1_macro": std_score
            })

            print(
                "depth =",
                depth,
                "| leaf =",
                leaf,
                "| F1 =",
                mean_score,
                "| std =",
                std_score
            )

    joblib.dump(
        rf_tuning_results,
        rf_tuning_path
    )

Résultats Random Forest chargés


In [88]:
rf_depth_results = pd.DataFrame(
    rf_tuning_results
)

rf_depth_results = (
    rf_depth_results
    .sort_values(
        "mean_f1_macro",
        ascending=False
    )
)

print(
    rf_depth_results.head(10)
)

    max_depth  min_samples_leaf  mean_f1_macro  std_f1_macro
16        NaN                 5       0.975667      0.001611
12       20.0                 5       0.975238      0.001894
17        NaN                10       0.974475      0.001786
8        16.0                 5       0.974318      0.001937
13       20.0                10       0.974175      0.001736
9        16.0                10       0.973284      0.001561
18        NaN                20       0.972332      0.001740
14       20.0                20       0.972213      0.001554
10       16.0                20       0.972018      0.001532
4        12.0                 5       0.971743      0.002436


#### Conclusion sur la profondeur et la taille minimale des feuilles

La validation temporelle montre que les meilleures performances sont obtenues avec des arbres relativement profonds et des feuilles de petite taille.

La meilleure moyenne est obtenue avec :

$$
max\_depth=None
$$

et :

$$
min\_samples\_leaf=5
$$

avec :

$$
F1_{macro}\approx0.97567
$$

Cependant, la configuration :

$$
max\_depth=20
$$

$$
min\_samples\_leaf=5
$$

obtient :

$$
F1_{macro}\approx0.97524
$$

L'écart entre les deux configurations est très faible par rapport à la variabilité observée entre les folds.

On retient donc :

$$
max\_depth=20,\qquad min\_samples\_leaf=5
$$

afin de conserver une régularisation explicite tout en maintenant une performance pratiquement équivalente au meilleur résultat.

In [89]:
max_features_values = [
    "sqrt",
    "log2",
    0.25,
    0.5,
    1.0
]

In [90]:
best_depth = 20
best_leaf = 5

In [91]:
rf_features_path = os.path.join(
    models_folder,
    "rf_max_features_tuning.joblib"
)

if os.path.exists(rf_features_path):

    rf_features_results = joblib.load(
        rf_features_path
    )

    print("Résultats max_features chargés")

else:

    rf_features_results = []

    for max_feat in max_features_values:

        fold_scores = []

        for train_idx, val_idx in tscv_rf.split(
            X_train_hd
        ):

            X_cv_train = X_train_hd.iloc[
                train_idx
            ]

            X_cv_val = X_train_hd.iloc[
                val_idx
            ]

            y_cv_train = y_train_hd.iloc[
                train_idx
            ]

            y_cv_val = y_train_hd.iloc[
                val_idx
            ]

            model_rf = RandomForestClassifier(
                n_estimators=100,
                max_depth=best_depth,
                min_samples_leaf=best_leaf,
                max_features=max_feat,
                n_jobs=-1,
                random_state=42
            )

            model_rf.fit(
                X_cv_train,
                y_cv_train
            )

            pred_cv = model_rf.predict(
                X_cv_val
            )

            score = f1_score(
                y_cv_val,
                pred_cv,
                average="macro"
            )

            fold_scores.append(score)

        rf_features_results.append({
            "max_features": max_feat,
            "mean_f1_macro": np.mean(
                fold_scores
            ),
            "std_f1_macro": np.std(
                fold_scores
            )
        })

        print(
            max_feat,
            np.mean(fold_scores),
            np.std(fold_scores)
        )

    joblib.dump(
        rf_features_results,
        rf_features_path
    )

Résultats max_features chargés


In [92]:
rf_features_df = pd.DataFrame(
    rf_features_results
).sort_values(
    "mean_f1_macro",
    ascending=False
)

print(rf_features_df)

  max_features  mean_f1_macro  std_f1_macro
3          0.5       0.981536      0.002245
2         0.25       0.980747      0.001902
4          1.0       0.980720      0.002356
0         sqrt       0.975238      0.001894
1         log2       0.967046      0.001615


#### Conclusion sur `max_features`

Le meilleur résultat est obtenu avec :

$$
max\_features = 0.5
$$

avec :

$$
F1_{macro} \approx 0.98154
$$

Cette valeur améliore nettement la configuration `"sqrt"` :

$$
0.98154 > 0.97524
$$

Dans ce problème, chaque arbre bénéficie donc de l'accès à une proportion plus importante des variables disponibles.

Pour la suite, on retient :

$$
max\_depth=20
$$

$$
min\_samples\_leaf=5
$$

$$
max\_features=0.5
$$

In [93]:
import time

"""n_values = [50, 100, 200, 400]

rf = RandomForestClassifier(
    warm_start=True,
    max_depth=20,
    min_samples_leaf=5,
    max_features=0.5,
    n_jobs=-1,
    random_state=42
)

results = []
start = time.time()
MAX_MINUTES = 5

for n in n_values:

    if (time.time() - start) / 60 > MAX_MINUTES:
        print("Arrêt : budget de temps atteint.")
        break

    rf.set_params(n_estimators=n)
    rf.fit(X_train_hd, y_train_hd)

    pred = rf.predict(X_val_hd)
    score = f1_score(
        y_val_hd,
        pred,
        average="macro"
    )

    results.append([n, score])

    joblib.dump(
        results,
        os.path.join(
            models_folder,
            "rf_n_estimators_progress.joblib"
        )
    )

    print(n, "arbres | F1 =", score)"""

'n_values = [50, 100, 200, 400]\n\nrf = RandomForestClassifier(\n    warm_start=True,\n    max_depth=20,\n    min_samples_leaf=5,\n    max_features=0.5,\n    n_jobs=-1,\n    random_state=42\n)\n\nresults = []\nstart = time.time()\nMAX_MINUTES = 5\n\nfor n in n_values:\n\n    if (time.time() - start) / 60 > MAX_MINUTES:\n        print("Arrêt : budget de temps atteint.")\n        break\n\n    rf.set_params(n_estimators=n)\n    rf.fit(X_train_hd, y_train_hd)\n\n    pred = rf.predict(X_val_hd)\n    score = f1_score(\n        y_val_hd,\n        pred,\n        average="macro"\n    )\n\n    results.append([n, score])\n\n    joblib.dump(\n        results,\n        os.path.join(\n            models_folder,\n            "rf_n_estimators_progress.joblib"\n        )\n    )\n\n    print(n, "arbres | F1 =", score)'

In [94]:
"""results_n = pd.DataFrame(
    results,
    columns=["n_estimators", "F1_macro"]
)

print(results_n)"""

'results_n = pd.DataFrame(\n    results,\n    columns=["n_estimators", "F1_macro"]\n)\n\nprint(results_n)'

In [95]:
rf_params = {
    "n_estimators": 50,
    "max_depth": 20,
    "min_samples_leaf": 5,
    "max_features": 0.5
}

#### Conclusion sur le nombre d'arbres

Le Random Forest avec 50 arbres obtient :

$$
F1_{macro} \approx 0.98187
$$

L'entraînement a atteint le budget de calcul fixé avant de pouvoir tester davantage d'arbres.

On retient donc :

$$
n\_estimators = 50
$$

comme compromis entre performance et coût computationnel.

L'objectif n'est pas nécessairement de rechercher un optimum extrêmement précis lorsque le coût de calcul devient disproportionné par rapport au gain potentiel.

#### C.4 Importance des variables

Le Random Forest permet d'estimer l'importance des variables à partir de la diminution d'impureté apportée par leurs splits.

Une variable fréquemment utilisée pour produire de fortes réductions d'impureté reçoit une importance élevée.

Les importances sont normalisées :

$$
\sum_{j=1}^{p} Importance_j = 1
$$

Ces valeurs indiquent l'utilité prédictive des variables dans le Random Forest, mais ne doivent pas être interprétées comme des effets causaux.

In [96]:
import time
from sklearn.ensemble import RandomForestClassifier

path = os.path.join(
    models_folder,
    "random_forest_tuned_50.joblib"
)

MAX_MINUTES = 5
TARGET = 50
STEP = 10

# Charger seulement si c'est vraiment un Random Forest
if os.path.exists(path):
    obj = joblib.load(path)
else:
    obj = None

if isinstance(obj, RandomForestClassifier):
    rf_final = obj
    print("Modèle récupéré :", rf_final.n_estimators, "arbres")

else:
    print("Ancien fichier invalide : reconstruction du modèle.")

    rf_final = RandomForestClassifier(
        n_estimators=STEP,
        max_depth=20,
        min_samples_leaf=5,
        max_features=0.5,
        warm_start=True,
        n_jobs=-1,
        random_state=42
    )

    rf_final.fit(
        X_train_hd,
        y_train_hd
    )

    joblib.dump(rf_final, path)

    print(STEP, "arbres sauvegardés")

Modèle récupéré : 50 arbres


In [97]:
start = time.time()

while rf_final.n_estimators < TARGET:

    if (time.time() - start) / 60 >= MAX_MINUTES:
        print("Arrêt : budget de temps atteint.")
        break

    new_n = min(
        rf_final.n_estimators + STEP,
        TARGET
    )

    rf_final.set_params(
        n_estimators=new_n
    )

    rf_final.fit(
        X_train_hd,
        y_train_hd
    )

    joblib.dump(
        rf_final,
        path
    )

    print(new_n, "arbres sauvegardés")

In [98]:
print(type(rf_final))
print(
    "Nombre d'arbres :",
    rf_final.n_estimators
)

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
Nombre d'arbres : 50


In [99]:
feature_importance_rf = pd.DataFrame({
    "feature": X_train_hd.columns,
    "importance": rf_final.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(feature_importance_rf.head(20))

             feature  importance
0              lag_1    0.550851
1              lag_2    0.164119
343   rolling_mean_6    0.068677
335          lag_336    0.040205
2              lag_3    0.025918
361           diff_1    0.022510
46            lag_47    0.016805
334          lag_335    0.015632
48            lag_49    0.013354
47            lag_48    0.009183
367      dev_mean_3h    0.003406
412  lag336_hour_sin    0.003296
45            lag_46    0.002725
333          lag_334    0.002597
355         hour_sin    0.002049
330          lag_331    0.001663
3              lag_4    0.001549
413  lag336_hour_cos    0.001423
331          lag_332    0.001199
356         hour_cos    0.001176


#### Conclusion sur l'importance des variables

L'analyse des importances montre que le Random Forest concentre une grande partie de son pouvoir prédictif sur un nombre relativement limité de variables.

La variable la plus importante est :

$$
lag_1
$$

avec une importance d'environ :

$$
55.1\%
$$

Elle est suivie notamment par :

$$
lag_2 \approx 16.4\%
$$

$$
rolling\_mean_6 \approx 6.9\%
$$

$$
lag_{336} \approx 4.0\%
$$

Les 20 premières variables représentent environ :

$$
94.8\%
$$

de l'importance totale.

Le modèle utilise donc principalement l'information de très court terme, complétée par la dynamique locale, la saisonnalité hebdomadaire et certaines variables calendaires.

Ces importances décrivent l'utilité prédictive des variables dans le Random Forest mais ne doivent pas être interprétées comme des effets causaux.

De plus, certaines variables étant fortement corrélées, l'importance peut être répartie entre plusieurs variables proches.

#### C.5 Modèle final et comparaison

Après l'étude de la profondeur, de la taille minimale des feuilles, du nombre de variables disponibles à chaque split et du nombre d'arbres, le Random Forest retenu utilise :

$$
n\_estimators = 50
$$

$$
max\_depth = 20
$$

$$
min\_samples\_leaf = 5
$$

$$
max\_features = 0.5
$$

Le modèle final est maintenant comparé au Decision Tree et à la régression logistique haute dimension.

In [100]:
# Aucun réentraînement
pred_rf_final = rf_final.predict(X_val_hd)

accuracy_rf_final = accuracy_score(
    y_val_hd, pred_rf_final
)

f1_rf_final = f1_score(
    y_val_hd,
    pred_rf_final,
    average="macro"
)

print("Accuracy RF final :", accuracy_rf_final)
print("F1 macro RF final :", f1_rf_final)

Accuracy RF final : 0.9818493150684932
F1 macro RF final : 0.981870789368509


In [101]:
conf_rf_final = confusion_matrix(
    y_val_hd,
    pred_rf_final
)

print(conf_rf_final)

[[5074   79    0]
 [ 123 8798   61]
 [   0   55 3330]]


In [102]:
comparison_step2 = pd.DataFrame({
    "Model": [
        "Logistic Regression HD",
        "Decision Tree",
        "Random Forest baseline",
        "Random Forest tuned"
    ],
    "Accuracy": [
        accuracy_hd,
        accuracy_tree,
        accuracy_rf,
        accuracy_rf_final
    ],
    "F1_macro": [
        f1_macro_hd,
        f1_macro_tree,
        f1_macro_rf,
        f1_rf_final
    ]
})

print(comparison_step2)

                    Model  Accuracy  F1_macro
0  Logistic Regression HD  0.988927  0.988930
1           Decision Tree  0.974600  0.974445
2  Random Forest baseline  0.975913  0.975610
3     Random Forest tuned  0.981849  0.981871


In [103]:
joblib.dump(
    comparison_step2,
    os.path.join(
        models_folder,
        "step2_final_comparison.joblib"
    )
)

['c:\\Users\\thiba\\OneDrive\\Bureau\\Projet energie\\models\\step2_final_comparison.joblib']

### Conclusion générale de l'Étape 2

Cette étape a comparé plusieurs méthodes de classification basées sur les arbres.

Le Decision Tree obtient :

$$
F1_{macro} \approx 0.9744
$$

Le Random Forest baseline améliore légèrement ce résultat :

$$
F1_{macro} \approx 0.9756
$$

Après optimisation guidée des hyperparamètres, le Random Forest atteint :

$$
F1_{macro} \approx 0.9819
$$

La meilleure configuration étudiée utilise notamment :

$$
max\_depth=20
$$

$$
min\_samples\_leaf=5
$$

$$
max\_features=0.5
$$

Le tuning permet donc un gain réel par rapport à la configuration initiale.

Cependant, la régression logistique haute dimension reste le meilleur modèle à ce stade :

$$
F1_{macro} \approx 0.9889
$$

L'analyse des importances du Random Forest montre également que la prédiction repose principalement sur l'information récente, en particulier `lag_1`, `lag_2`, ainsi que certaines statistiques glissantes et variables saisonnières.

Les 20 variables les plus importantes représentent environ :

$$
94.8\%
$$

de l'importance totale.

Cette étape montre qu'un modèle non linéaire peut être amélioré par un réglage raisonné de sa complexité, mais que l'ingénierie des variables reste déterminante dans ce problème.

## Étape 3 — Gradient Boosting Methods
### Méthodes de boosting pour la classification

### Partie A — Mise en pratique

In [104]:
from sklearn.ensemble import HistGradientBoostingClassifier
import time

boost_path = os.path.join(
    models_folder,
    "hist_gradient_boosting.joblib"
)

MAX_MINUTES = 5

In [105]:
if os.path.exists(boost_path):

    hgb = joblib.load(boost_path)
    print("Modèle chargé")

else:

    start = time.time()

    hgb = HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_iter=50,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    hgb.fit(
        X_train_hd,
        y_train_hd
    )

    joblib.dump(
        hgb,
        boost_path
    )

    print(
        "Temps :",
        (time.time() - start) / 60,
        "minutes"
    )

Modèle chargé


In [106]:
pred_hgb = hgb.predict(
    X_val_hd
)

accuracy_hgb = accuracy_score(
    y_val_hd,
    pred_hgb
)

f1_hgb = f1_score(
    y_val_hd,
    pred_hgb,
    average="macro"
)

print(
    "Accuracy HGB :",
    accuracy_hgb
)

print(
    "F1 macro HGB :",
    f1_hgb
)

Accuracy HGB : 0.984703196347032
F1 macro HGB : 0.984646826450562


In [107]:
comparison_boosting = pd.DataFrame({
    "Model": [
        "Logistic Regression HD",
        "Random Forest tuned",
        "HistGradientBoosting"
    ],
    "F1_macro": [
        f1_macro_hd,
        f1_rf_final,
        f1_hgb
    ]
})

print(comparison_boosting)

                    Model  F1_macro
0  Logistic Regression HD  0.988930
1     Random Forest tuned  0.981871
2    HistGradientBoosting  0.984647


#### Conclusion de la baseline HistGradientBoosting

Le HistGradientBoosting obtient :

$$
F1_{macro} \approx 0.98465
$$

Il améliore le Random Forest optimisé :

$$
0.98465 > 0.98187
$$

mais reste inférieur à la régression logistique haute dimension :

$$
0.98465 < 0.98893
$$

Le boosting semble donc mieux exploiter les relations non linéaires que le Random Forest dans ce problème, sans encore surpasser le modèle linéaire enrichi.

#### A.2 XGBoost Classifier

XGBoost construit les arbres séquentiellement afin de corriger progressivement les erreurs du modèle.

Il ajoute également plusieurs mécanismes de régularisation et d'optimisation numérique.

Afin de contrôler le coût computationnel, l'entraînement est limité à cinq minutes.

In [108]:
import time
from xgboost import XGBClassifier
from xgboost.callback import TrainingCallback

class TimeLimit(TrainingCallback):
    def __init__(self, seconds=300):
        self.seconds = seconds

    def before_training(self, model):
        self.start = time.time()
        return model

    def after_iteration(self, model, epoch, evals_log):
        return time.time() - self.start >= self.seconds

In [109]:
xgb_path = os.path.join(
    models_folder,
    "xgboost_block2.joblib"
)

if os.path.exists(xgb_path):
    xgb = joblib.load(xgb_path)
    print("XGBoost chargé")

else:
    xgb = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="mlogloss",
        n_jobs=-1,
        random_state=42,
        callbacks=[TimeLimit(300)]
    )

    xgb.fit(
        X_train_hd,
        y_train_hd
    )

    joblib.dump(xgb, xgb_path)
    print("XGBoost sauvegardé")

XGBoost chargé


In [110]:
pred_xgb = xgb.predict(X_val_hd)

accuracy_xgb = accuracy_score(
    y_val_hd,
    pred_xgb
)

f1_xgb = f1_score(
    y_val_hd,
    pred_xgb,
    average="macro"
)

print("Accuracy XGBoost :", accuracy_xgb)
print("F1 macro XGBoost :", f1_xgb)

Accuracy XGBoost : 0.9845890410958904
F1 macro XGBoost : 0.9845228485314911


In [111]:
comparison_boosting = pd.DataFrame({
    "Model": [
        "Logistic Regression HD",
        "Random Forest tuned",
        "HistGradientBoosting",
        "XGBoost"
    ],
    "F1_macro": [
        f1_macro_hd,
        f1_rf_final,
        f1_hgb,
        f1_xgb
    ]
})

print(comparison_boosting)

                    Model  F1_macro
0  Logistic Regression HD  0.988930
1     Random Forest tuned  0.981871
2    HistGradientBoosting  0.984647
3                 XGBoost  0.984523


#### Conclusion de la comparaison des modèles de boosting

Les deux méthodes de boosting obtiennent des performances très proches :

$$
F1_{macro,\ HistGradientBoosting} \approx 0.98465
$$

$$
F1_{macro,\ XGBoost} \approx 0.98452
$$

L'écart entre les deux modèles est très faible :

$$
\Delta F1 \approx 0.00012
$$

À ce stade, aucun des deux modèles de boosting ne surpasse la régression logistique haute dimension :

$$
F1_{macro,\ Logistic} \approx 0.98893
$$

Le HistGradientBoosting présente même une très légère avance sur XGBoost dans cette configuration initiale.

Ces résultats seront utilisés comme références avant les améliorations guidées par la théorie.

### Partie B — Théorie et formules importantes

#### B.1 Principe général du boosting

Le boosting construit plusieurs modèles faibles de manière séquentielle.

Contrairement au Random Forest, où les arbres sont construits relativement indépendamment, chaque nouvel arbre du boosting cherche à corriger les erreurs commises par les arbres précédents.

On construit progressivement un modèle :

$$
F_M(x)
=
F_0(x)
+
\sum_{m=1}^{M}
\eta f_m(x)
$$

où :

- $F_0(x)$ est le modèle initial,
- $f_m(x)$ est le nouvel arbre ajouté à l'itération $m$,
- $M$ est le nombre total d'arbres,
- $\eta$ est le learning rate.

Le modèle devient donc progressivement plus complexe.

#### B.2 Gradient Boosting

Le Gradient Boosting généralise l'idée de correction des erreurs.

On considère une fonction de perte :

$$
L(y,F(x))
$$

À chaque itération, on cherche à améliorer le modèle dans la direction qui réduit le plus cette perte.

On calcule alors le pseudo-résidu :

$$
r_{im}
=
-
\left[
\frac{\partial L(y_i,F(x_i))}
{\partial F(x_i)}
\right]_{F=F_{m-1}}
$$

Le nouvel arbre est entraîné pour approximer ces pseudo-résidus.

Le modèle est ensuite mis à jour :

$$
F_m(x)
=
F_{m-1}(x)
+
\eta f_m(x)
$$

Le boosting peut donc être interprété comme une forme de descente de gradient dans l'espace des fonctions.

#### B.3 Learning rate et nombre d'arbres

Le paramètre `learning_rate` contrôle la contribution de chaque nouvel arbre.

Dans :

$$
F_m(x)
=
F_{m-1}(x)
+
\eta f_m(x)
$$

$\eta$ représente le learning rate.

Un petit learning rate signifie que chaque arbre apporte une petite correction.

On obtient généralement :

$$
\eta \downarrow
\quad\Rightarrow\quad
M \uparrow
$$

Autrement dit, un petit learning rate nécessite généralement davantage d'arbres.

À l'inverse, un learning rate élevé permet d'apprendre plus rapidement mais peut conduire à une adaptation trop agressive aux données.

Il existe donc un compromis entre :

$$
learning\ rate
\leftrightarrow
nombre\ d'arbres
$$

#### B.4 Profondeur des arbres

Dans un modèle de boosting, les arbres sont généralement utilisés comme apprenants faibles.

Un arbre peu profond apprend des corrections simples.

Une profondeur plus importante permet de modéliser davantage d'interactions entre les variables mais augmente également la complexité du modèle.

On retrouve donc un compromis biais-variance :

$$
profondeur \uparrow
\Rightarrow
biais \downarrow
$$

mais potentiellement :

$$
variance \uparrow
$$

Le boosting n'a donc pas nécessairement besoin d'arbres extrêmement profonds.

#### B.5 Random Forest vs Boosting

Le Random Forest et le boosting combinent tous les deux plusieurs arbres, mais leur fonctionnement est très différent.

Random Forest :

$$
Arbre_1,\ Arbre_2,\ ...,\ Arbre_B
$$

sont construits de manière largement indépendante puis agrégés.

Son objectif principal est de réduire la variance.

Le boosting construit au contraire :

$$
Arbre_1
\rightarrow
Arbre_2
\rightarrow
Arbre_3
\rightarrow ...
$$

Chaque nouvel arbre dépend du modèle déjà construit.

Son objectif est de corriger progressivement les erreurs et de réduire principalement le biais, tout en contrôlant la variance grâce à la régularisation.

#### B.6 Pourquoi HistGradientBoosting est rapide

Les méthodes classiques d'arbres doivent rechercher de nombreux seuils possibles pour chaque variable.

Les méthodes basées sur des histogrammes regroupent les valeurs continues dans un nombre limité d'intervalles.

Au lieu d'étudier chaque valeur possible, l'algorithme travaille sur ces groupes.

Cela réduit fortement le coût de recherche des meilleurs splits.

Cette approche est particulièrement utile lorsque le nombre d'observations est élevé, comme dans notre jeu de données énergétique.

#### B.7 XGBoost et régularisation

XGBoost reprend le principe du Gradient Boosting mais ajoute plusieurs mécanismes de régularisation.

L'objectif général peut s'écrire sous la forme :

$$
\mathcal{L}
=
\sum_{i=1}^{n}
L(y_i,\hat{y}_i)
+
\sum_{m=1}^{M}
\Omega(f_m)
$$

où :

- $L$ mesure l'erreur de prédiction,
- $\Omega(f_m)$ pénalise la complexité des arbres.

Une forme simplifiée de cette pénalisation est :

$$
\Omega(f)
=
\gamma T
+
\frac{\lambda}{2}
\sum_{j=1}^{T} w_j^2
$$

où :

- $T$ est le nombre de feuilles,
- $w_j$ est la valeur associée à la feuille $j$,
- $\gamma$ pénalise la création de nouvelles feuilles,
- $\lambda$ contrôle la régularisation L2.

XGBoost cherche donc à améliorer les prédictions tout en évitant de créer des arbres inutilement complexes.


#### B.8 Subsampling et stochastic boosting

Le boosting peut également utiliser seulement une partie des observations ou des variables à chaque itération.

Avec :

`subsample = 0.8`

chaque nouvel arbre utilise environ 80 % des observations disponibles.

Avec :

`colsample_bytree = 0.8`

chaque arbre utilise environ 80 % des variables.

Cette randomisation peut réduire le surapprentissage.

Elle introduit davantage de diversité entre les arbres et peut améliorer la généralisation.

On retrouve une idée proche du Random Forest, mais les arbres restent entraînés séquentiellement.


#### B.9 Overfitting et Early Stopping

Lorsque le nombre d'arbres augmente, l'erreur d'entraînement diminue généralement.

Cependant, après un certain point, les performances de validation peuvent cesser de s'améliorer.

On peut alors observer :

$$
Erreur_{train} \downarrow
$$

mais :

$$
Erreur_{validation} \uparrow
$$

ce qui indique du surapprentissage.

L'early stopping consiste à arrêter l'entraînement lorsque les performances de validation ne s'améliorent plus pendant plusieurs itérations.

On peut représenter le principe par :

$$
m^*
=
\arg\min_m
L_{validation}(m)
$$

Le modèle conserve alors le nombre d'itérations correspondant à la meilleure performance de validation.

Cela permet à la fois :

- de limiter le surapprentissage,
- de réduire le temps de calcul,
- d'éviter de choisir inutilement un nombre très élevé d'arbres.


#### B.10 Comparaison théorique des modèles étudiés

Les quatre grandes approches étudiées jusqu'à présent ont des comportements différents.

**Régression logistique**

Elle apprend une frontière linéaire dans l'espace des variables.

Elle est particulièrement efficace lorsque le feature engineering rend les classes presque linéairement séparables.

Dans notre problème, elle obtient actuellement :

$$
F1_{macro} \approx 0.98893
$$


**Decision Tree**

Il construit une succession de règles non linéaires.

Il est très flexible mais possède généralement une variance élevée.

Résultat :

$$
F1_{macro} \approx 0.97445
$$


**Random Forest**

Il combine plusieurs arbres grâce au bagging et à la randomisation des variables.

Son objectif principal est de réduire la variance des arbres individuels.

Après tuning :

$$
F1_{macro} \approx 0.98187
$$


**Gradient Boosting**

Il construit les arbres séquentiellement afin de corriger progressivement les erreurs du modèle précédent.

Il permet généralement de réduire davantage le biais qu'un Random Forest.

HistGradientBoosting obtient :

$$
F1_{macro} \approx 0.98465
$$

et XGBoost :

$$
F1_{macro} \approx 0.98452
$$


#### B.11 Interprétation biais-variance

On peut résumer grossièrement les comportements par :

Decision Tree :

$$
Biais \ relativement\ faible
$$

$$
Variance \ élevée
$$

Random Forest :

$$
Variance \downarrow
$$

grâce au bagging.

Boosting :

$$
Biais \downarrow
$$

grâce à la correction successive des erreurs.

Cependant, un boosting trop complexe peut également augmenter la variance.

La régularisation, le learning rate, la profondeur et l'early stopping jouent donc un rôle essentiel.


#### B.12 Importance du learning rate

Le learning rate contrôle la vitesse à laquelle le modèle corrige ses erreurs.

Un grand learning rate :

$$
\eta \uparrow
$$

permet d'apprendre rapidement, mais augmente le risque de corrections trop agressives.

Un petit learning rate :

$$
\eta \downarrow
$$

produit généralement des corrections plus progressives et plus stables, mais nécessite davantage d'arbres.

On cherche donc un compromis :

$$
petit\ learning\ rate
+
davantage\ d'arbres
$$

contre :

$$
grand\ learning\ rate
+
moins\ d'arbres
$$


#### Conclusion de la Partie B

Le boosting construit progressivement un modèle en ajoutant des arbres destinés à réduire la fonction de perte.

Il peut être interprété comme une forme de descente de gradient dans l'espace des fonctions.

Ses performances dépendent principalement :

- du learning rate,
- du nombre d'arbres,
- de la profondeur des arbres,
- de la régularisation,
- du sous-échantillonnage des observations et des variables,
- de l'early stopping.

XGBoost ajoute plusieurs mécanismes de régularisation au Gradient Boosting classique, tandis que HistGradientBoosting utilise une représentation par histogrammes afin de réduire fortement le coût computationnel.

Dans notre problème, les modèles de boosting améliorent le Random Forest mais restent actuellement légèrement derrière la régression logistique haute dimension.

La Partie C cherchera donc à déterminer si un réglage ciblé des paramètres peut réduire cet écart, tout en respectant un budget de calcul maximal.

### Partie C — Améliorations guidées par la théorie

#### C.1 Learning rate et nombre d'itérations

Le learning rate et le nombre d'arbres sont fortement liés.

Un learning rate faible nécessite généralement davantage d'itérations, tandis qu'un learning rate plus élevé permet d'apprendre plus rapidement.

On cherche donc un compromis entre performance et coût de calcul.

Le réglage est effectué sur une validation temporelle interne au train afin de conserver l'année 2025 comme validation indépendante.

In [112]:
cut = int(0.8 * len(X_train_hd))

X_fit = X_train_hd.iloc[:cut]
y_fit = y_train_hd.iloc[:cut]

X_dev = X_train_hd.iloc[cut:]
y_dev = y_train_hd.iloc[cut:]

In [113]:
configs = [
    (0.20, 30),
    (0.10, 50),
    (0.05, 100),
    (0.03, 150)
]

In [114]:
import time

path = os.path.join(
    models_folder,
    "hgb_lr_iter_results.joblib"
)

results = []
start = time.time()
MAX_SECONDS = 300

for lr, n_iter in configs:

    if time.time() - start >= MAX_SECONDS:
        print("Arrêt : budget de 5 minutes atteint.")
        break

    model = HistGradientBoostingClassifier(
        learning_rate=lr,
        max_iter=n_iter,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    model.fit(X_fit, y_fit)

    pred = model.predict(X_dev)

    score = f1_score(
        y_dev,
        pred,
        average="macro"
    )

    results.append([lr, n_iter, score])

    joblib.dump(results, path)

    print(
        "lr =", lr,
        "| iter =", n_iter,
        "| F1 =", score
    )

lr = 0.2 | iter = 30 | F1 = 0.981375258868741
lr = 0.1 | iter = 50 | F1 = 0.9821472077214661
lr = 0.05 | iter = 100 | F1 = 0.9821215447155969
lr = 0.03 | iter = 150 | F1 = 0.982382831522823


In [115]:
results_hgb = pd.DataFrame(
    results,
    columns=[
        "learning_rate",
        "max_iter",
        "F1_macro"
    ]
).sort_values(
    "F1_macro",
    ascending=False
)

print(results_hgb)

   learning_rate  max_iter  F1_macro
3           0.03       150  0.982383
1           0.10        50  0.982147
2           0.05       100  0.982122
0           0.20        30  0.981375


In [116]:
hgb_tuned = HistGradientBoostingClassifier(
    learning_rate=0.10,
    max_iter=50,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

#### Conclusion sur le learning rate

Les différentes combinaisons testées donnent des performances très proches.

La meilleure performance interne est obtenue avec :

$$
learning\_rate=0.03,\qquad max\_iter=150
$$

avec :

$$
F1_{macro}\approx0.98238
$$

Cependant, la configuration :

$$
learning\_rate=0.10,\qquad max\_iter=50
$$

obtient :

$$
F1_{macro}\approx0.98215
$$

L'écart avec la meilleure configuration n'est que d'environ :

$$
0.00024
$$

De plus, cette configuration nécessite seulement 50 itérations.

On retient donc :

$$
learning\_rate=0.10,\qquad max\_iter=50
$$

comme meilleur compromis entre performance et coût computationnel.

#### C.2 Validation du HistGradientBoosting retenu

Le compromis retenu est :

$$
learning\_rate = 0.10
$$

$$
max\_iter = 50
$$

Le modèle est maintenant réentraîné sur l'ensemble des données d'entraînement puis évalué une seule fois sur la validation temporelle 2025.

L'early stopping interne est désactivé afin de conserver entièrement le protocole de validation temporelle défini dans le projet.

In [117]:
hgb_final_path = os.path.join(
    models_folder,
    "hgb_lr010_iter50.joblib"
)

if os.path.exists(hgb_final_path):
    hgb_final = joblib.load(hgb_final_path)

else:
    hgb_final = HistGradientBoostingClassifier(
        learning_rate=0.10,
        max_iter=50,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42
    )

    hgb_final.fit(
        X_train_hd,
        y_train_hd
    )

    joblib.dump(
        hgb_final,
        hgb_final_path
    )

In [118]:
pred_hgb_final = hgb_final.predict(
    X_val_hd
)

accuracy_hgb_final = accuracy_score(
    y_val_hd,
    pred_hgb_final
)

f1_hgb_final = f1_score(
    y_val_hd,
    pred_hgb_final,
    average="macro"
)

print("Accuracy :", accuracy_hgb_final)
print("F1 macro :", f1_hgb_final)
print("Itérations :", hgb_final.n_iter_)

Accuracy : 0.9852739726027397
F1 macro : 0.9853398166763162
Itérations : 50


In [119]:
comparison_hgb = pd.DataFrame({
    "Model": [
        "Logistic Regression HD",
        "Random Forest tuned",
        "HGB baseline",
        "HGB final"
    ],
    "F1_macro": [
        f1_macro_hd,
        f1_rf_final,
        f1_hgb,
        f1_hgb_final
    ]
})

print(comparison_hgb)

                    Model  F1_macro
0  Logistic Regression HD  0.988930
1     Random Forest tuned  0.981871
2            HGB baseline  0.984647
3               HGB final  0.985340


#### Conclusion sur l'optimisation du HistGradientBoosting

Le HistGradientBoosting final atteint :

$$
F1_{macro} \approx 0.98534
$$

contre :

$$
F1_{macro} \approx 0.98465
$$

pour la baseline.

L'amélioration est réelle mais modérée.

La régression logistique haute dimension reste le meilleur modèle :

$$
F1_{macro} \approx 0.98893
$$

Compte tenu du faible écart entre les différentes configurations testées et du coût computationnel supplémentaire d'un tuning plus poussé, le HistGradientBoosting est considéré comme suffisamment optimisé à ce stade.

In [120]:
xgb_results = []
start = time.time()

for depth in [3, 5, 7]:

    if time.time() - start >= 300:
        print("Arrêt : limite de 5 minutes.")
        break

    model = XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=depth,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_fit, y_fit)

    score = f1_score(
        y_dev,
        model.predict(X_dev),
        average="macro"
    )

    xgb_results.append([depth, score])

    joblib.dump(
        xgb_results,
        os.path.join(
            models_folder,
            "xgb_depth_results.joblib"
        )
    )

    print(depth, score)

3 0.9694190818869544
5 0.9765015952012078
7 0.9793551255138365


In [121]:
xgb_depth_df = pd.DataFrame(
    xgb_results,
    columns=["max_depth", "F1_macro"]
).sort_values(
    "F1_macro",
    ascending=False
)

print(xgb_depth_df)

   max_depth  F1_macro
2          7  0.979355
1          5  0.976502
0          3  0.969419


#### Conclusion sur la profondeur de XGBoost

La profondeur des arbres influence fortement les performances de XGBoost.

Les résultats obtenus sur la validation temporelle interne sont :

$$
max\_depth=3 \rightarrow F1_{macro}\approx0.9694
$$

$$
max\_depth=5 \rightarrow F1_{macro}\approx0.9765
$$

$$
max\_depth=7 \rightarrow F1_{macro}\approx0.9794
$$

Parmi les valeurs testées, la meilleure profondeur est donc :

$$
\boxed{max\_depth=7}
$$

L'augmentation de la profondeur permet ici de capturer davantage d'interactions entre les variables.

Ces scores servent uniquement au choix de l'hyperparamètre et ne sont pas directement comparables aux scores obtenus sur la validation finale 2025.

In [122]:
xgb_final_path = os.path.join(
    models_folder,
    "xgboost_depth7_final.joblib"
)

if os.path.exists(xgb_final_path):

    xgb_final = joblib.load(xgb_final_path)

else:

    xgb_final = XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="mlogloss",
        n_jobs=-1,
        random_state=42
    )

    xgb_final.fit(
        X_train_hd,
        y_train_hd
    )

    joblib.dump(
        xgb_final,
        xgb_final_path
    )

In [123]:
pred_xgb_final = xgb_final.predict(X_val_hd)

accuracy_xgb_final = accuracy_score(
    y_val_hd,
    pred_xgb_final
)

f1_xgb_final = f1_score(
    y_val_hd,
    pred_xgb_final,
    average="macro"
)

print("Accuracy :", accuracy_xgb_final)
print("F1 macro :", f1_xgb_final)

Accuracy : 0.9824771689497717
F1 macro : 0.9825416463214567


In [124]:
comparison_step3 = pd.DataFrame({
    "Model": [
        "Logistic Regression HD",
        "Random Forest tuned",
        "HGB final",
        "XGBoost baseline",
        "XGBoost tuned"
    ],
    "F1_macro": [
        f1_macro_hd,
        f1_rf_final,
        f1_hgb_final,
        f1_xgb,
        f1_xgb_final
    ]
})

print(comparison_step3)

                    Model  F1_macro
0  Logistic Regression HD  0.988930
1     Random Forest tuned  0.981871
2               HGB final  0.985340
3        XGBoost baseline  0.984523
4           XGBoost tuned  0.982542


In [125]:
print(
    "XGB baseline :",
    xgb.get_booster().num_boosted_rounds()
)

print(
    "XGB tuned :",
    xgb_final.get_booster().num_boosted_rounds()
)

XGB baseline : 242
XGB tuned : 100


#### Conclusion sur l'optimisation de XGBoost

La validation temporelle interne suggérait qu'une profondeur plus importante améliorait les performances, avec :

$$
max\_depth=7
$$

comme meilleure valeur parmi celles testées.

Cependant, après évaluation sur la validation indépendante 2025, le modèle modifié obtient :

$$
F1_{macro} \approx 0.98254
$$

contre :

$$
F1_{macro} \approx 0.98452
$$

pour la baseline XGBoost.

L'amélioration observée sur la validation interne ne s'est donc pas généralisée à l'année 2025.

Ce résultat illustre l'importance de conserver une validation temporelle indépendante : un hyperparamètre performant sur une période particulière n'est pas nécessairement optimal sur une période future.

De plus, les deux configurations diffèrent également par leur nombre d'arbres, ce qui empêche d'attribuer entièrement l'écart à la seule profondeur.

Aucun tuning supplémentaire coûteux n'est effectué à ce stade.

### Conclusion générale de l'Étape 3

Cette étape a étudié les méthodes de boosting pour la classification de la demande énergétique.

Les performances finales sont :

$$
F1_{macro,\ Logistic} \approx 0.98893
$$

$$
F1_{macro,\ HGB} \approx 0.98534
$$

$$
F1_{macro,\ XGBoost} \approx 0.98452
$$

$$
F1_{macro,\ RandomForest} \approx 0.98187
$$

Le HistGradientBoosting est donc le meilleur modèle non linéaire étudié jusqu'à présent.

L'optimisation du HistGradientBoosting a apporté une légère amélioration, tandis que le tuning testé pour XGBoost n'a pas amélioré la généralisation sur 2025.

La régression logistique haute dimension conserve néanmoins la meilleure performance globale.

Ce résultat confirme une observation importante du projet : un feature engineering riche peut rendre un modèle relativement simple extrêmement compétitif face à des méthodes d'ensemble plus complexes.

L'étape montre également qu'un gain observé lors du tuning ne doit jamais être considéré comme acquis avant une évaluation sur une période temporelle indépendante.

Enfin, compte tenu du coût computationnel, les améliorations sont désormais évaluées conjointement selon deux critères :

$$
Performance
\quad\text{et}\quad
Coût\ computationnel
$$

## Étape 4 — Class Imbalance & Cost-Sensitive Learning
### Déséquilibre des classes et apprentissage sensible aux coûts

### Partie A — Mise en pratique

#### A.1 Analyse du déséquilibre des classes

Les trois classes ne possèdent pas exactement la même fréquence.

Sur l'entraînement, les seuils construits à partir des quantiles donnent approximativement :

$$
P(Y=0)\approx25\%
$$

$$
P(Y=1)\approx50\%
$$

$$
P(Y=2)\approx25\%
$$

Sur la validation 2025, la distribution évolue et la classe de forte consommation devient moins fréquente.

Cette situation permet d'étudier l'impact du déséquilibre des classes et des coûts de décision sur les prédictions.

In [126]:
print("Train :")
print(y_train_hd.value_counts(normalize=True))

print("\nValidation :")
print(y_val_hd.value_counts(normalize=True))

Train :
demand_class
1    0.500005
2    0.249998
0    0.249998
Name: proportion, dtype: float64

Validation :
demand_class
1    0.512671
0    0.294121
2    0.193208
Name: proportion, dtype: float64


In [127]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_hd
)

class_weights = dict(
    zip(classes, weights)
)

print(class_weights)

{np.int64(0): np.float64(1.3333460288442007), np.int64(1): np.float64(0.6666603190321126), np.int64(2): np.float64(1.3333460288442007)}


In [128]:
weight_array = np.array([
    class_weights[0],
    class_weights[1],
    class_weights[2]
])

In [129]:
weighted_proba = (
    proba_val_hd
    * weight_array
)

pred_weighted = np.argmax(
    weighted_proba,
    axis=1
)

In [130]:
f1_weighted_decision = f1_score(
    y_val_hd,
    pred_weighted,
    average="macro"
)

accuracy_weighted_decision = accuracy_score(
    y_val_hd,
    pred_weighted
)

print(
    "Accuracy décision standard :",
    accuracy_hd
)

print(
    "Accuracy décision pondérée :",
    accuracy_weighted_decision
)

print(
    "F1 standard :",
    f1_macro_hd
)

print(
    "F1 pondéré :",
    f1_weighted_decision
)

Accuracy décision standard : 0.9889269406392694
Accuracy décision pondérée : 0.9856735159817351
F1 standard : 0.9889302570044536
F1 pondéré : 0.9858159291879772


In [131]:
print(
    classification_report(
        y_val_hd,
        pred_weighted,
        target_names=[
            "Faible",
            "Intermédiaire",
            "Forte"
        ]
    )
)

               precision    recall  f1-score   support

       Faible       0.97      1.00      0.98      5153
Intermédiaire       0.99      0.98      0.99      8982
        Forte       0.98      0.99      0.99      3385

     accuracy                           0.99     17520
    macro avg       0.98      0.99      0.99     17520
 weighted avg       0.99      0.99      0.99     17520



In [132]:
print(
    confusion_matrix(
        y_val_hd,
        pred_weighted
    )
)

[[5132   21    0]
 [ 144 8776   62]
 [   0   24 3361]]


#### Conclusion sur la pondération automatique des classes

La pondération automatique des classes réduit légèrement les performances globales :

$$
F1_{macro,\ standard} \approx 0.98893
$$

$$
F1_{macro,\ pondéré} \approx 0.98582
$$

Cependant, elle améliore la détection de la classe de forte consommation.

Le nombre de fortes consommations incorrectement classées comme intermédiaires passe de :

$$
38
$$

à :

$$
24
$$

soit 14 périodes de forte consommation supplémentaires correctement détectées.

En contrepartie, le nombre de périodes intermédiaires classées à tort comme fortes augmente.

La pondération illustre donc un compromis entre :

$$
Precision
\leftrightarrow
Recall
$$

Une pondération n'est pas nécessairement destinée à maximiser l'accuracy ou le F1 global. Elle peut être pertinente lorsque certaines erreurs sont opérationnellement plus coûteuses que d'autres.

#### A.4 Décision sensible au coût des fortes consommations

Dans une application énergétique, toutes les erreurs ne possèdent pas nécessairement le même coût.

En particulier, ne pas détecter une période réellement caractérisée par une forte consommation peut être plus problématique qu'une fausse alerte.

On introduit donc une matrice de coût afin de modifier la règle de décision sans réentraîner le modèle.

In [133]:
cost_matrix_energy = np.array([
    [0, 1, 6],
    [1, 0, 3],
    [2, 1, 0]
])

In [134]:
expected_cost_energy = (
    proba_val_hd
    @ cost_matrix_energy.T
)

pred_energy_cost = np.argmin(
    expected_cost_energy,
    axis=1
)

In [135]:
print(
    classification_report(
        y_val_hd,
        pred_energy_cost,
        target_names=[
            "Faible",
            "Intermédiaire",
            "Forte"
        ]
    )
)

print(
    confusion_matrix(
        y_val_hd,
        pred_energy_cost
    )
)

print(
    "F1 macro :",
    f1_score(
        y_val_hd,
        pred_energy_cost,
        average="macro"
    )
)

               precision    recall  f1-score   support

       Faible       0.99      0.99      0.99      5153
Intermédiaire       0.99      0.98      0.99      8982
        Forte       0.97      1.00      0.98      3385

     accuracy                           0.99     17520
    macro avg       0.98      0.99      0.99     17520
 weighted avg       0.99      0.99      0.99     17520

[[5107   46    0]
 [  77 8805  100]
 [   0   12 3373]]
F1 macro : 0.9862002498698189


In [136]:
missed_high = np.sum(
    (y_val_hd.values == 2)
    &
    (pred_energy_cost != 2)
)

false_high = np.sum(
    (y_val_hd.values != 2)
    &
    (pred_energy_cost == 2)
)

print("Fortes consommations ratées :", missed_high)
print("Fausses alertes fortes :", false_high)

Fortes consommations ratées : 12
Fausses alertes fortes : 100


In [137]:
decision_comparison = pd.DataFrame({
    "Decision": [
        "Standard",
        "Balanced",
        "Energy cost"
    ],
    "F1_macro": [
        f1_macro_hd,
        f1_weighted_decision,
        0.9862002498698189
    ],
    "Missed_high": [
        38,
        24,
        12
    ],
    "False_high": [
        33,
        62,
        100
    ]
})

print(decision_comparison)

      Decision  F1_macro  Missed_high  False_high
0     Standard  0.988930           38          33
1     Balanced  0.985816           24          62
2  Energy cost  0.986200           12         100


#### Conclusion sur la décision sensible au coût

La modification de la fonction de décision permet de réduire fortement le nombre de périodes de forte consommation non détectées.

Avec la règle standard, le modèle ratait :

$$
38
$$

périodes de forte consommation.

La pondération automatique réduit ce nombre à :

$$
24
$$

et la décision spécifiquement adaptée au risque énergétique le réduit à :

$$
12
$$

En contrepartie, le nombre de fausses alertes de forte consommation augmente :

$$
33
\rightarrow
62
\rightarrow
100
$$

pour les trois règles de décision.

Le F1 macro diminue légèrement :

$$
0.98893
\rightarrow
0.98620
$$

mais cette diminution globale permet d'obtenir un rappel presque parfait pour la classe de forte consommation.

Cette expérience montre qu'en pratique, la meilleure règle de décision ne dépend pas nécessairement de l'accuracy ou du F1 maximal.

Elle dépend également du coût associé aux différentes erreurs.

Dans un contexte où manquer un épisode de forte consommation est particulièrement coûteux, une décision sensible au coût peut donc être préférable malgré une légère diminution des performances globales.

#### A.5 Courbe Precision-Recall pour les fortes consommations

La règle de décision peut être modifiée en faisant varier le seuil nécessaire pour déclencher une alerte de forte consommation.

Diminuer le seuil permet généralement de détecter davantage de vraies fortes consommations, mais augmente le nombre de fausses alertes.

On étudie donc le compromis entre précision et rappel pour la classe 2.

In [138]:
from sklearn.metrics import precision_recall_curve, average_precision_score

y_high = (y_val_hd.values == 2).astype(int)
p_high = proba_val_hd[:, 2]

precision_pr, recall_pr, thresholds_pr = precision_recall_curve(
    y_high,
    p_high
)

ap_high = average_precision_score(
    y_high,
    p_high
)

print("Average Precision :", ap_high)

Average Precision : 0.9996513266738452


In [139]:
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:

    pred_high = (p_high >= t).astype(int)

    missed = np.sum(
        (y_high == 1) & (pred_high == 0)
    )

    false_alerts = np.sum(
        (y_high == 0) & (pred_high == 1)
    )

    print(
        t,
        "| ratées =", missed,
        "| fausses alertes =", false_alerts
    )

0.3 | ratées = 17 | fausses alertes = 81
0.4 | ratées = 27 | fausses alertes = 49
0.5 | ratées = 38 | fausses alertes = 33
0.6 | ratées = 54 | fausses alertes = 19
0.7 | ratées = 70 | fausses alertes = 10


#### Conclusion sur le seuil de détection

La classe de forte consommation est extrêmement bien séparée des autres classes.

L'Average Precision obtenue est :

$$
AP \approx 0.99965
$$

ce qui indique une excellente capacité du modèle à classer les périodes réellement fortes au-dessus des autres observations.

Le choix du seuil produit cependant un compromis clair.

Avec un seuil de :

$$
0.30
$$

seules 17 fortes consommations sont ratées, mais 81 fausses alertes sont générées.

À l'inverse, avec :

$$
0.70
$$

le nombre de fausses alertes tombe à 10, mais 70 fortes consommations ne sont plus détectées.

Le seuil de décision ne doit donc pas être choisi uniquement pour maximiser une métrique statistique.

Il dépend du coût relatif entre :

$$
\text{rater une forte consommation}
$$

et :

$$
\text{déclencher une fausse alerte}
$$

Ces résultats sur la validation 2025 servent ici à analyser le comportement du modèle. Un seuil opérationnel définitif devra être choisi à partir des données d'entraînement afin de préserver l'indépendance de la validation.

### Partie B — Théorie et formules importantes

#### B.1 Déséquilibre des classes

Un problème de classification est déséquilibré lorsque les différentes classes n'apparaissent pas avec les mêmes fréquences.

Si :

$$
P(Y=k)
$$

est très faible pour une classe $k$, un modèle peut obtenir une bonne accuracy globale tout en détectant mal cette classe.

C'est pourquoi l'accuracy seule peut être insuffisante dans un problème déséquilibré.

Dans notre problème, les classes ne sont pas extrêmement déséquilibrées, mais la classe de forte consommation est moins fréquente sur la validation 2025 que la classe intermédiaire.


#### B.2 Class weights

Une méthode classique consiste à attribuer davantage de poids aux observations provenant de classes rares.

Une pondération équilibrée peut être définie par :

$$
w_k
=
\frac{n}
{K n_k}
$$

où :

- $n$ est le nombre total d'observations,
- $K$ est le nombre de classes,
- $n_k$ est le nombre d'observations de la classe $k$.

Ainsi :

$$
n_k \downarrow
\quad\Rightarrow\quad
w_k \uparrow
$$

Les erreurs commises sur une classe rare deviennent donc plus coûteuses pendant l'apprentissage ou la prise de décision.


#### B.3 Precision et Recall

Pour une classe considérée comme positive :

$$
Precision
=
\frac{TP}{TP+FP}
$$

La précision répond à :

> Parmi toutes les alertes émises, combien sont correctes ?

Le rappel est :

$$
Recall
=
\frac{TP}{TP+FN}
$$

Il répond à :

> Parmi tous les vrais événements, combien ont été détectés ?

Dans notre problème énergétique :

- un faux positif correspond à une fausse alerte de forte consommation ;
- un faux négatif correspond à une forte consommation réellement présente mais non détectée.


#### B.4 Compromis Precision-Recall

Modifier le seuil de décision modifie simultanément la précision et le rappel.

Lorsque le seuil diminue :

$$
Recall \uparrow
$$

mais généralement :

$$
Precision \downarrow
$$

Lorsque le seuil augmente :

$$
Precision \uparrow
$$

mais généralement :

$$
Recall \downarrow
$$

Il n'existe donc pas nécessairement un seuil universellement optimal.

Le seuil dépend de l'objectif opérationnel du système.

#### B.5 Courbe Precision-Recall

La courbe Precision-Recall représente la précision obtenue pour différents niveaux de rappel lorsque le seuil de décision varie.

Elle est particulièrement utile lorsque la classe positive est relativement rare.

Contrairement à une seule valeur de seuil, elle permet d'étudier l'ensemble des compromis possibles entre :

$$
Precision
$$

et :

$$
Recall
$$


#### B.6 Average Precision

L'Average Precision résume la courbe Precision-Recall en une seule valeur.

De manière intuitive, elle mesure la capacité du modèle à attribuer des scores élevés aux observations positives.

Une valeur proche de :

$$
1
$$

indique une séparation presque parfaite entre observations positives et négatives.

Dans notre problème :

$$
AP \approx 0.99965
$$

pour la détection des fortes consommations.

Le modèle possède donc une excellente capacité de classement de cette classe.

#### B.7 Fonction de coût

Dans une classification classique, toutes les erreurs sont souvent considérées de la même manière.

Cependant, dans une application réelle, certaines erreurs peuvent être plus graves que d'autres.

On introduit alors une fonction de coût :

$$
C(a,k)
$$

où :

- $a$ est la classe prédite,
- $k$ est la vraie classe.

Par exemple, dans notre problème énergétique, on peut considérer qu'une forte consommation non détectée est plus coûteuse qu'une fausse alerte.

Une matrice de coût possible est :

$$
C=
\begin{pmatrix}
0 & 1 & 6 \\
1 & 0 & 3 \\
2 & 1 & 0
\end{pmatrix}
$$

Les lignes correspondent aux décisions prises et les colonnes aux vraies classes.

Ainsi :

$$
C(1,2)=3
$$

signifie que prédire une consommation intermédiaire alors que la consommation est réellement forte possède un coût de 3.

À l'inverse :

$$
C(2,1)=1
$$

signifie qu'une fausse alerte de forte consommation est considérée comme moins grave.


#### B.8 Risque conditionnel et règle de décision de Bayes

Supposons que le modèle fournisse les probabilités :

$$
P(Y=k|X=x)
$$

Pour une décision possible $a$, on calcule son coût moyen attendu :

$$
R(a|x)
=
\sum_{k=0}^{K-1}
C(a,k)
P(Y=k|X=x)
$$

Cette quantité est appelée risque conditionnel.

La décision optimale au sens de la théorie de la décision est alors :

$$
\hat{y}(x)
=
\arg\min_a
R(a|x)
$$

On ne choisit donc plus nécessairement la classe ayant la probabilité maximale.

On choisit la décision qui minimise le coût attendu.

La règle classique :

$$
\hat y
=
\arg\max_k P(Y=k|X)
$$

est un cas particulier correspondant essentiellement à une situation où toutes les mauvaises classifications possèdent le même coût.


#### B.9 Seuil optimal dans le cas binaire

Considérons uniquement la détection d'une forte consommation :

$$
Y=
\begin{cases}
1 & \text{forte consommation} \\
0 & \text{sinon}
\end{cases}
$$

On note :

$$
p=P(Y=1|X)
$$

Supposons que :

- $C_{FN}$ soit le coût d'une forte consommation non détectée ;
- $C_{FP}$ soit le coût d'une fausse alerte.

Si on prédit la classe positive, le coût attendu est :

$$
R(1|x)
=
C_{FP}(1-p)
$$

Si on prédit la classe négative :

$$
R(0|x)
=
C_{FN}p
$$

On choisit la classe positive lorsque :

$$
C_{FP}(1-p)
<
C_{FN}p
$$

En développant :

$$
C_{FP}
<
p(C_{FP}+C_{FN})
$$

donc :

$$
p
>
\frac{C_{FP}}
{C_{FP}+C_{FN}}
$$

Le seuil optimal théorique est donc :

$$
t^*
=
\frac{C_{FP}}
{C_{FP}+C_{FN}}
$$

Cette formule montre que le seuil de $0.5$ n'est pas toujours optimal.


#### Exemple

Si une forte consommation ratée coûte trois fois plus qu'une fausse alerte :

$$
C_{FN}=3
$$

et :

$$
C_{FP}=1
$$

alors :

$$
t^*
=
\frac{1}{1+3}
=
0.25
$$

Le modèle doit alors déclencher une alerte dès que :

$$
P(Y=\text{Forte}|X)>0.25
$$

Cela augmente généralement le rappel au prix d'une diminution de la précision.


#### B.10 Interprétation de nos résultats

Nos expériences montrent directement ce compromis.

Avec le seuil standard :

$$
t=0.5
$$

nous obtenons :

$$
38
$$

fortes consommations ratées et :

$$
33
$$

fausses alertes.

Avec :

$$
t=0.3
$$

nous obtenons seulement :

$$
17
$$

fortes consommations ratées mais :

$$
81
$$

fausses alertes.

À l'inverse, avec :

$$
t=0.7
$$

le nombre de fausses alertes tombe à :

$$
10
$$

mais :

$$
70
$$

fortes consommations sont ratées.

Le seuil contrôle donc directement le compromis entre la sécurité de détection et le nombre de fausses alertes.


#### B.11 Pondération des classes vs modification du seuil

Deux approches différentes peuvent être utilisées.

La première consiste à modifier l'apprentissage avec des poids de classes.

Les erreurs associées aux classes rares ou importantes reçoivent alors davantage de poids pendant l'entraînement.

La seconde consiste à conserver le modèle probabiliste puis à modifier uniquement la règle de décision.

Cette deuxième approche présente un avantage important :

le même modèle probabiliste peut être utilisé pour différentes politiques de décision.

Par exemple, un opérateur prudent peut utiliser un seuil faible afin de maximiser le rappel, tandis qu'un autre opérateur peut utiliser un seuil plus élevé afin de limiter les fausses alertes.

La prédiction probabiliste et la décision opérationnelle sont donc deux étapes distinctes.


#### B.12 Pourquoi le F1 maximal n'est pas toujours la meilleure décision

Une métrique comme le F1 cherche un compromis statistique entre précision et rappel :

$$
F1
=
2
\frac{Precision \times Recall}
{Precision+Recall}
$$

Mais cette formule ne contient aucune information sur le coût réel des erreurs.

Deux systèmes possédant le même F1 peuvent avoir des conséquences opérationnelles très différentes.

Dans un problème énergétique, si une forte consommation non détectée est beaucoup plus coûteuse qu'une fausse alerte, il peut être rationnel d'accepter une légère diminution du F1 afin d'augmenter fortement le rappel.

La meilleure décision dépend donc de :

$$
Performance\ statistique
+
Coût\ des\ erreurs
$$


#### Conclusion de la Partie B

Le déséquilibre des classes ne doit pas être étudié uniquement à partir de leur fréquence.

Il faut également considérer l'importance opérationnelle des différentes erreurs.

La théorie de la décision permet de formaliser ce problème grâce à une matrice de coût.

La décision optimale devient :

$$
\hat{y}(x)
=
\arg\min_a
\sum_k
C(a,k)P(Y=k|X=x)
$$

Dans le cas binaire, le coût des faux positifs et des faux négatifs permet également de déterminer un seuil théorique :

$$
t^*
=
\frac{C_{FP}}
{C_{FP}+C_{FN}}
$$

Nos expériences montrent clairement qu'abaisser le seuil de détection des fortes consommations réduit fortement le nombre d'événements ratés, mais augmente le nombre de fausses alertes.

La Partie C utilisera cette théorie pour rechercher une règle de décision adaptée au problème énergétique sans réentraîner inutilement les modèles.

### Partie C — Améliorations guidées par la théorie

#### C.1 Sélection temporelle du seuil de forte consommation

Les expériences précédentes ont montré qu'abaisser le seuil permet de détecter davantage de périodes de forte consommation au prix d'un plus grand nombre de fausses alertes.

Le seuil opérationnel ne doit cependant pas être choisi directement sur la validation 2025.

On utilise donc une validation temporelle interne aux données d'entraînement :

$$
Train_{fit}
\rightarrow
Train_{dev}
\rightarrow
Validation\ 2025
$$

On suppose ici qu'une forte consommation ratée possède un coût trois fois supérieur à une fausse alerte :

$$
C_{FN}=3
$$

$$
C_{FP}=1
$$

In [140]:
internal_path = os.path.join(
    models_folder,
    "logistic_hd_internal_threshold.joblib"
)

if os.path.exists(internal_path):
    logistic_internal = joblib.load(internal_path)

else:
    logistic_internal = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=1,
            max_iter=2000
        ))
    ])

    logistic_internal.fit(X_fit, y_fit)
    joblib.dump(logistic_internal, internal_path)

In [141]:
proba_dev = logistic_internal.predict_proba(
    X_dev
)

p_high_dev = proba_dev[:, 2]
y_high_dev = (y_dev.values == 2).astype(int)

In [142]:
thresholds = np.arange(
    0.10,
    0.71,
    0.05
)

results_threshold = []

for t in thresholds:

    pred = (p_high_dev >= t)

    missed = np.sum(
        (y_high_dev == 1) & (~pred)
    )

    false_alerts = np.sum(
        (y_high_dev == 0) & pred
    )

    cost = (
        3 * missed
        + false_alerts
    )

    results_threshold.append([
        t,
        missed,
        false_alerts,
        cost
    ])

threshold_df = pd.DataFrame(
    results_threshold,
    columns=[
        "threshold",
        "missed_high",
        "false_high",
        "cost"
    ]
)

print(
    threshold_df.sort_values("cost")
)

    threshold  missed_high  false_high  cost
3        0.25           25         219   294
5        0.35           53         155   314
4        0.30           45         184   319
2        0.20           16         274   322
6        0.40           66         129   327
7        0.45           75         111   336
1        0.15            9         319   346
8        0.50           92          90   366
9        0.55          113          69   408
0        0.10            4         402   414
10       0.60          132          54   450
11       0.65          156          43   511
12       0.70          191          30   603


In [143]:
best_threshold = (
    threshold_df
    .sort_values("cost")
    .iloc[0]["threshold"]
)

print(
    "Seuil retenu :",
    best_threshold
)

Seuil retenu : 0.25000000000000006


In [144]:
joblib.dump(
    threshold_df,
    os.path.join(
        models_folder,
        "high_demand_threshold_results.joblib"
    )
)

['c:\\Users\\thiba\\OneDrive\\Bureau\\Projet energie\\models\\high_demand_threshold_results.joblib']

In [145]:
print(
    "Seuil théorique :",
    0.25
)

print(
    "Seuil empirique :",
    best_threshold
)

Seuil théorique : 0.25
Seuil empirique : 0.25000000000000006


In [146]:
pred_cost_final = np.where(
    proba_val_hd[:, 2]
    >= best_threshold,
    2,
    np.argmax(
        proba_val_hd[:, :2],
        axis=1
    )
)

In [147]:
f1_cost_final = f1_score(
    y_val_hd,
    pred_cost_final,
    average="macro"
)

missed_final = np.sum(
    (y_val_hd.values == 2)
    &
    (pred_cost_final != 2)
)

false_final = np.sum(
    (y_val_hd.values != 2)
    &
    (pred_cost_final == 2)
)

print("F1 macro :", f1_cost_final)
print("Fortes ratées :", missed_final)
print("Fausses alertes :", false_final)

F1 macro : 0.9862002498698189
Fortes ratées : 12
Fausses alertes : 100


In [148]:
comparison_cost = pd.DataFrame({
    "Rule": [
        "Standard",
        "Cost-sensitive"
    ],
    "F1_macro": [
        f1_macro_hd,
        f1_cost_final
    ],
    "Missed_high": [
        38,
        missed_final
    ],
    "False_high": [
        33,
        false_final
    ]
})

print(comparison_cost)

             Rule  F1_macro  Missed_high  False_high
0        Standard   0.98893           38          33
1  Cost-sensitive   0.98620           12         100


#### Conclusion sur la sélection du seuil

La théorie de la décision prédit, pour :

$$
C_{FN}=3
$$

et :

$$
C_{FP}=1
$$

un seuil optimal :

$$
t^*
=
\frac{1}{1+3}
=
0.25
$$

La recherche empirique effectuée uniquement sur la validation temporelle interne retrouve exactement :

$$
t_{empirique}=0.25
$$

Cette concordance constitue une validation intéressante de la règle de décision théorique.

Sur la validation indépendante 2025, la règle standard avec un seuil de 0.5 produit :

$$
38
$$

fortes consommations ratées et :

$$
33
$$

fausses alertes.

Avec le seuil sensible au coût de 0.25 :

$$
12
$$

fortes consommations sont ratées et :

$$
100
$$

fausses alertes sont générées.

Avec la fonction de coût :

$$
Cost=3FN+FP
$$

le coût total passe de :

$$
147
$$

à :

$$
136
$$

soit une réduction d'environ :

$$
7.5\%
$$

Le F1 macro diminue légèrement, mais la politique de décision atteint mieux l'objectif opérationnel défini.

Cette expérience illustre donc la différence entre :

$$
optimisation\ d'une\ métrique\ statistique
$$

et :

$$
optimisation\ d'une\ décision\ opérationnelle.
$$

In [149]:
cost_ratios = [1, 2, 3, 5, 10]

results_cost = []

for c_fn in cost_ratios:

    t = 1 / (1 + c_fn)

    pred = np.where(
        proba_val_hd[:, 2] >= t,
        2,
        np.argmax(
            proba_val_hd[:, :2],
            axis=1
        )
    )

    missed = np.sum(
        (y_val_hd.values == 2)
        & (pred != 2)
    )

    false = np.sum(
        (y_val_hd.values != 2)
        & (pred == 2)
    )

    results_cost.append([
        c_fn,
        t,
        missed,
        false,
        c_fn * missed + false
    ])

cost_sensitivity = pd.DataFrame(
    results_cost,
    columns=[
        "C_FN",
        "threshold",
        "missed_high",
        "false_high",
        "total_cost"
    ]
)

print(cost_sensitivity)

   C_FN  threshold  missed_high  false_high  total_cost
0     1   0.500000           38          33          71
1     2   0.333333           24          62         110
2     3   0.250000           12         100         136
3     5   0.166667            7         136         171
4    10   0.090909            1         178         188


#### Conclusion sur la sensibilité au coût

L'analyse de sensibilité montre que la politique de décision dépend directement du coût attribué aux fortes consommations non détectées.

Lorsque le coût d'un faux négatif augmente, le seuil théorique diminue :

$$
t^*
=
\frac{C_{FP}}
{C_{FP}+C_{FN}}
$$

Par exemple :

$$
C_{FN}=1
\Rightarrow
t^*=0.50
$$

$$
C_{FN}=3
\Rightarrow
t^*=0.25
$$

$$
C_{FN}=10
\Rightarrow
t^*\approx0.091
$$

Lorsque le seuil diminue, le nombre de fortes consommations ratées chute fortement :

$$
38
\rightarrow
24
\rightarrow
12
\rightarrow
7
\rightarrow
1
$$

mais le nombre de fausses alertes augmente :

$$
33
\rightarrow
62
\rightarrow
100
\rightarrow
136
\rightarrow
178
$$

Il n'est cependant pas pertinent de comparer directement le coût total entre différentes valeurs de $C_{FN}$, puisque la fonction de coût elle-même change.

Le rapport de coût doit être fixé à partir du besoin opérationnel.

Une fois ce coût défini, la théorie de la décision permet d'en déduire automatiquement une règle de décision cohérente.

### Conclusion générale de l'Étape 4

Cette étape a montré qu'un problème de classification ne doit pas être évalué uniquement à partir de l'accuracy ou du F1-score.

La pondération automatique des classes réduit légèrement les performances globales, mais augmente le rappel des classes minoritaires.

Une approche plus pertinente consiste à associer explicitement un coût aux différentes erreurs.

Pour la détection des fortes consommations, la théorie donne dans le cas binaire :

$$
t^*
=
\frac{C_{FP}}
{C_{FP}+C_{FN}}
$$

Pour :

$$
C_{FN}=3,\qquad C_{FP}=1
$$

le seuil théorique est :

$$
t^*=0.25
$$

La validation temporelle interne retrouve exactement ce seuil.

Sur l'année 2025, cette règle permet de réduire les fortes consommations ratées de :

$$
38
\rightarrow12
$$

au prix d'une augmentation des fausses alertes :

$$
33
\rightarrow100
$$

Le coût défini par :

$$
3FN+FP
$$

passe néanmoins de :

$$
147
$$

à :

$$
136
$$

soit une réduction d'environ :

$$
7.5\%
$$

Cette étape met donc en évidence une distinction fondamentale entre :

$$
Prédiction
$$

et :

$$
Décision
$$

Un modèle probabiliste peut rester identique tandis que sa règle de décision est adaptée aux contraintes opérationnelles.

## Étape 5 — Uncertainty-Aware Classification
### Classification avec prise en compte de l'incertitude

### Partie A — Mise en pratique

#### A.1 Mesurer l'incertitude des prédictions

Un modèle peut prédire la bonne classe tout en étant plus ou moins certain de sa décision.

On étudie trois quantités :

- la probabilité maximale,
- l'entropie prédictive,
- la marge entre les deux classes les plus probables.

In [150]:
confidence = proba_val_hd.max(axis=1)

entropy = -np.sum(
    proba_val_hd * np.log(proba_val_hd + 1e-12),
    axis=1
) / np.log(3)

sorted_proba = np.sort(proba_val_hd, axis=1)

margin = (
    sorted_proba[:, -1]
    - sorted_proba[:, -2]
)

correct = (
    pred_val_hd == y_val_hd.values
)

In [151]:
uncertainty_df = pd.DataFrame({
    "confidence": confidence,
    "entropy": entropy,
    "margin": margin,
    "correct": correct
})

print(
    uncertainty_df.groupby("correct")[
        ["confidence", "entropy", "margin"]
    ].mean()
)

         confidence   entropy    margin
correct                                
False      0.667123  0.548723  0.334246
True       0.987093  0.030228  0.974187


#### A.2 Incertitude des erreurs

Si les erreurs du modèle possèdent en moyenne une confiance plus faible, une entropie plus élevée et une marge plus faible que les bonnes prédictions, alors l'incertitude peut être utilisée pour détecter les situations difficiles.

In [152]:
errors_uncertainty = uncertainty_df[
    ~uncertainty_df["correct"]
].sort_values("confidence")

print(
    errors_uncertainty.head(10)
)

       confidence   entropy    margin  correct
12109    0.500658  0.630929  0.001317    False
5424     0.500668  0.630929  0.001335    False
7720     0.501074  0.630928  0.002148    False
16292    0.502379  0.630919  0.004759    False
6779     0.502830  0.630915  0.005660    False
15889    0.503031  0.630913  0.006061    False
15708    0.504324  0.630896  0.008648    False
6510     0.505117  0.630882  0.010235    False
12595    0.506450  0.630854  0.012900    False
10247    0.506700  0.630848  0.013401    False


#### Conclusion sur l'incertitude

Les erreurs du modèle sont nettement plus incertaines que les bonnes prédictions.

Pour les prédictions correctes :

$$
Confidence \approx 0.987
$$

alors que pour les erreurs :

$$
Confidence \approx 0.667
$$

Les erreurs présentent également une entropie plus élevée et une marge beaucoup plus faible.

L'incertitude constitue donc un bon indicateur pour détecter les observations difficiles.

### Partie C — Analyse et améliorations guidées par la théorie

#### C.1 Reject option : permettre au modèle de s'abstenir

Au lieu de forcer une prédiction lorsque le modèle est très incertain, on peut lui permettre de s'abstenir.

La règle devient :

$$
\text{prédire seulement si }
\max_k P(Y=k|X) \geq t
$$

Sinon, la prédiction est envoyée vers une analyse supplémentaire.

On cherche alors un compromis entre :

$$
Coverage
$$

et :

$$
Accuracy
$$

In [153]:
confidence_dev = proba_dev.max(axis=1)

pred_dev = logistic_internal.predict(X_dev)

thresholds = [0.70, 0.80, 0.90, 0.95, 0.99]

reject_results = []

for t in thresholds:

    keep = confidence_dev >= t

    reject_results.append([
        t,
        keep.mean(),
        accuracy_score(
            y_dev[keep],
            pred_dev[keep]
        )
    ])

reject_df = pd.DataFrame(
    reject_results,
    columns=[
        "threshold",
        "coverage",
        "selective_accuracy"
    ]
)

print(reject_df)

   threshold  coverage  selective_accuracy
0       0.70  0.980766            0.994515
1       0.80  0.967436            0.997195
2       0.90  0.948274            0.998795
3       0.95  0.930302            0.999437
4       0.99  0.889883            0.999840


#### C.2 Sélection d'un seuil d'abstention

La validation temporelle interne montre qu'un seuil de confiance de :

$$
t=0.90
$$

permet de conserver environ :

$$
94.8\%
$$

des observations tout en atteignant une accuracy d'environ :

$$
99.88\%
$$

On retient donc ce seuil comme compromis entre couverture et fiabilité.

In [154]:
reject_threshold = 0.90

confidence_val = proba_val_hd.max(axis=1)

keep = confidence_val >= reject_threshold

coverage_val = keep.mean()

selective_accuracy_val = accuracy_score(
    y_val_hd.values[keep],
    pred_val_hd[keep]
)

n_rejected = (~keep).sum()

print("Coverage :", coverage_val)
print("Selective accuracy :", selective_accuracy_val)
print("Observations rejetées :", n_rejected)

Coverage : 0.9482305936073059
Selective accuracy : 0.9995786432312045
Observations rejetées : 907


In [155]:
errors = pred_val_hd != y_val_hd.values

total_errors = errors.sum()

errors_rejected = np.sum(
    errors & (~keep)
)

print("Erreurs totales :", total_errors)
print("Erreurs captées par le rejet :", errors_rejected)

print(
    "Part des erreurs captées :",
    errors_rejected / total_errors
)

Erreurs totales : 194
Erreurs captées par le rejet : 187
Part des erreurs captées : 0.9639175257731959


#### C.3 Conclusion sur l'abstention

Le seuil de confiance :

$$
t=0.90
$$

permet de conserver :

$$
94.82\%
$$

des observations.

Sur ces prédictions acceptées, l'accuracy atteint :

$$
99.96\%
$$

Le modèle rejette seulement :

$$
5.18\%
$$

des observations, mais ces observations contiennent :

$$
96.4\%
$$

de toutes les erreurs.

Sur les 194 erreurs initiales, 187 sont donc identifiées comme des situations incertaines.

Il ne reste que 7 erreurs parmi les prédictions considérées comme suffisamment fiables.

Cette expérience montre que l'incertitude estimée par les probabilités du modèle permet d'identifier efficacement les situations difficiles.

Un système réel pourrait donc utiliser deux niveaux de décision :

$$
Confidence \geq 0.90
\Rightarrow
\text{prédiction automatique}
$$

$$
Confidence < 0.90
\Rightarrow
\text{analyse supplémentaire ou abstention}
$$

### Conclusion générale de l'Étape 5

Les probabilités produites par le modèle permettent de mesurer efficacement son incertitude.

Les erreurs présentent une confiance beaucoup plus faible, une entropie plus élevée et une marge plus faible que les bonnes prédictions.

Une règle d'abstention avec un seuil de confiance de 0.90 permet de détecter 96.4 % des erreurs tout en conservant environ 94.8 % des observations.

Le modèle peut donc non seulement prédire une classe, mais également identifier les situations dans lesquelles sa prédiction est moins fiable.

## Étape 6 — Ensembling & Final Model Selection
### Combinaison des modèles et sélection finale

### Partie A — Mise en pratique

#### A.1 Soft Voting

Les modèles étudiés n'apprennent pas exactement les mêmes frontières de décision.

On peut donc moyenner leurs probabilités afin de vérifier si leurs erreurs se compensent.

Aucun nouvel entraînement n'est nécessaire.

In [156]:
proba_log = logistic_hd.predict_proba(X_val_hd)
proba_hgb = hgb_final.predict_proba(X_val_hd)
proba_rf = rf_final.predict_proba(X_val_hd)
proba_xgb = xgb.predict_proba(X_val_hd)

In [157]:
ensembles = {
    "Equal voting": (
        proba_log + proba_hgb +
        proba_rf + proba_xgb
    ) / 4,

    "Logistic + HGB": (
        0.5 * proba_log
        + 0.5 * proba_hgb
    ),

    "Logistic dominant": (
        0.7 * proba_log
        + 0.3 * proba_hgb
    )
}

In [158]:
ensemble_results = []

for name, proba in ensembles.items():

    pred = np.argmax(proba, axis=1)

    ensemble_results.append([
        name,
        accuracy_score(y_val_hd, pred),
        f1_score(
            y_val_hd,
            pred,
            average="macro"
        )
    ])

ensemble_df = pd.DataFrame(
    ensemble_results,
    columns=[
        "Model",
        "Accuracy",
        "F1_macro"
    ]
)

print(ensemble_df)

               Model  Accuracy  F1_macro
0       Equal voting  0.987215  0.987149
1     Logistic + HGB  0.988756  0.988674
2  Logistic dominant  0.989212  0.989217


In [159]:
final_models = pd.concat([
    pd.DataFrame({
        "Model": [
            "Logistic Regression HD",
            "HGB final",
            "Random Forest tuned",
            "XGBoost baseline"
        ],
        "F1_macro": [
            f1_macro_hd,
            f1_hgb_final,
            f1_rf_final,
            f1_xgb
        ]
    }),
    ensemble_df[
        ["Model", "F1_macro"]
    ]
])

print(
    final_models.sort_values(
        "F1_macro",
        ascending=False
    )
)

                    Model  F1_macro
2       Logistic dominant  0.989217
0  Logistic Regression HD  0.988930
1          Logistic + HGB  0.988674
0            Equal voting  0.987149
1               HGB final  0.985340
3        XGBoost baseline  0.984523
2     Random Forest tuned  0.981871


In [160]:
joblib.dump(
    final_models,
    os.path.join(
        models_folder,
        "block2_final_model_comparison.joblib"
    )
)

['c:\\Users\\thiba\\OneDrive\\Bureau\\Projet energie\\models\\block2_final_model_comparison.joblib']

### Partie C — Analyse et sélection finale

#### C.1 Complémentarité des erreurs

Le meilleur ensemble combine les probabilités de la régression logistique et du HistGradientBoosting :

$$
P_{ensemble}
=
0.7P_{Logistic}
+
0.3P_{HGB}
$$

L'objectif est de vérifier si le HistGradientBoosting corrige certaines erreurs commises par la régression logistique.

In [161]:
proba_final = (
    0.7 * proba_log
    + 0.3 * proba_hgb
)

pred_final = np.argmax(
    proba_final,
    axis=1
)

err_log = pred_val_hd != y_val_hd.values
err_hgb = (
    hgb_final.predict(X_val_hd)
    != y_val_hd.values
)
err_final = pred_final != y_val_hd.values

print("Erreurs Logistic :", err_log.sum())
print("Erreurs HGB      :", err_hgb.sum())
print("Erreurs Ensemble :", err_final.sum())

print(
    "Logistic fausse / HGB correcte :",
    np.sum(err_log & ~err_hgb)
)

print(
    "Logistic correcte / HGB faux :",
    np.sum(~err_log & err_hgb)
)

print(
    "Les deux faux :",
    np.sum(err_log & err_hgb)
)

Erreurs Logistic : 194
Erreurs HGB      : 258
Erreurs Ensemble : 189
Logistic fausse / HGB correcte : 91
Logistic correcte / HGB faux : 155
Les deux faux : 103


In [162]:
print(
    confusion_matrix(
        y_val_hd,
        pred_final
    )
)

print(
    classification_report(
        y_val_hd,
        pred_final,
        target_names=[
            "Faible",
            "Intermédiaire",
            "Forte"
        ]
    )
)

[[5101   52    0]
 [  68 8880   34]
 [   0   35 3350]]
               precision    recall  f1-score   support

       Faible       0.99      0.99      0.99      5153
Intermédiaire       0.99      0.99      0.99      8982
        Forte       0.99      0.99      0.99      3385

     accuracy                           0.99     17520
    macro avg       0.99      0.99      0.99     17520
 weighted avg       0.99      0.99      0.99     17520



In [163]:
block2_final_config = {
    "model": "Soft Voting",
    "components": [
        "Logistic Regression HD",
        "HistGradientBoosting"
    ],
    "weights": [0.7, 0.3],
    "validation_f1_macro": 0.989217,
    "test_2026_used": False
}

joblib.dump(
    block2_final_config,
    os.path.join(
        models_folder,
        "block2_final_config.joblib"
    )
)

['c:\\Users\\thiba\\OneDrive\\Bureau\\Projet energie\\models\\block2_final_config.joblib']

#### C.2 Sélection finale

Parmi les modèles étudiés, la meilleure performance sur la validation 2025 est obtenue par un soft voting combinant :

$$
70\%
$$

de régression logistique haute dimension et :

$$
30\%
$$

de HistGradientBoosting.

Le F1 macro passe de :

$$
0.988930
$$

pour la régression logistique seule à :

$$
0.989217
$$

pour l'ensemble.

Le gain est faible mais montre que les deux modèles possèdent une certaine complémentarité.

À l'inverse, le vote uniforme entre tous les modèles réduit les performances :

$$
F1_{macro}\approx0.98715
$$

Un ensemble n'est donc utile que si les modèles apportent une information complémentaire pertinente.

Le candidat final du Bloc 2 est donc :

$$
\boxed{
P_{final}
=
0.7P_{Logistic}
+
0.3P_{HGB}
}
$$

Le jeu de test 2026 reste volontairement intact et ne sera utilisé qu'après verrouillage global du projet.

### Conclusion de l'Étape 6

Le HistGradientBoosting et la régression logistique commettent des erreurs partiellement différentes.

Le HGB corrige 91 erreurs de la régression logistique, mais introduit également 155 erreurs sur des observations correctement classées par celle-ci.

Le soft voting :

$$
P_{final}
=
0.7P_{Logistic}
+
0.3P_{HGB}
$$

permet néanmoins de réduire le nombre total d'erreurs :

$$
194 \rightarrow 189
$$

et atteint :

$$
F1_{macro}=0.989217
$$

contre :

$$
0.988930
$$

pour la régression logistique seule.

Le gain est faible mais réel.

Aucune confusion directe entre les classes Faible et Forte n'est observée.

Le soft voting Logistic + HGB est donc retenu comme candidat final du Bloc 2.

# Conclusion du Bloc 2

Ce bloc a permis de construire et comparer plusieurs méthodes de classification :

- régression logistique haute dimension,
- arbres et Random Forest,
- HistGradientBoosting,
- XGBoost,
- classification sensible aux coûts,
- estimation de l'incertitude,
- abstention,
- ensembling.

Le meilleur modèle individuel reste la régression logistique haute dimension.

Le meilleur résultat global est obtenu avec un soft voting :

$$
70\% \ Logistic
+
30\% \ HGB
$$

avec :

$$
F1_{macro}=0.989217
$$

La classification probabiliste permet également d'adapter les décisions aux coûts métier et d'identifier efficacement les prédictions incertaines.

Le test final 2026 reste intact.

In [164]:
print(pd.infer_freq(X.index))
print(X.index[:5])

NameError: name 'X' is not defined